# 13m — Diagnóstico de sobreajuste e de mudança entre os dias dos modelos multiclasse (RF e SVM)

**Pergunta do TCC:** os quatro modelos multiclasse (RF padrão, RF ajustado, SVM padrão e SVM ajustado) sofreram de sobreajuste? E a queda de desempenho do mesmo dia (validação) para o outro dia (teste) vem de **decorar o treino**, ou de o **dado ter mudado**, e **quais atributos mudaram**? Este notebook não treina nenhum modelo para uso: ele **mede sinais** nos modelos que já existem, sem alterar nenhum arquivo deles.

### Dois tipos de sobreajuste (e por que precisamos separar)
1. **Decorar o treino (clássico):** o modelo acerta as linhas que viu e erra linhas novas *do mesmo tipo*. A validação e a validação cruzada, que vêm do mesmo dia do treino, **detectam** isso.
2. **Ajustar-se ao dia do treino:** o modelo acerta dados do mesmo dia, mas cai em outro dia (o teste). A validação do mesmo dia **não detecta** isso; só o teste revela a queda, e a causa só aparece **comparando os dados dos dois dias**.

### O que o notebook faz
| Etapa | O que mede | Responde a |
|---|---|---|
| **2.1** Treino × validação cruzada × validação × teste | desempenho do modelo nas linhas de treino, na CV, na validação e no teste (os números da CV e do teste vêm dos arquivos já gravados) | tipo 1 e tipo 2 |
| **3.1** Curva de aprendizagem | desempenho com 10%, 25%, 50% e 100% do ajuste | os dados bastam? a lacuna se fecha? |
| **4.1** Embaralhamento de rótulos | treina com rótulos embaralhados e avalia na validação real: o ROC-AUC deve ficar em ~0,5 | vazamento ou erro no processo |
| **4.2** Linhas repetidas entre ajuste e validação | quantas linhas da validação têm vetor **idêntico** ao de uma linha do ajuste (e quantas com rótulo diferente) | a validação do mesmo dia é **otimista**? |
| **5.1** Validação adversarial, por classe | um classificador tenta distinguir linhas do **dia do treino** de linhas do **dia do teste**, classe a classe; ROC-AUC ~0,5 = sem mudança, ~1 = os dois dias são diferentes. Mostra os **atributos** que mais mudam | tipo 2: **o dado mudou? quais atributos?** |
| **6.1** Veredito | aplica as regras abaixo e escreve um texto para o TCC | conclusão |

### Regras de decisão (convenções, gravadas em `regras_decisao.json` antes de qualquer cálculo)
| Evidência | pequena / ok | moderada | grande / alerta |
|---|---|---|---|
| **Q1 — decorar o treino** | | | |
| lacuna treino − validação, acurácia balanceada | < 0,01 | 0,01 a 0,03 | > 0,03 |
| lacuna treino − validação, ROC-AUC (macro) | < 0,005 | 0,005 a 0,02 | > 0,02 |
| embaralhamento de rótulos (ROC-AUC macro na validação real) | média das repetições dentro de 0,5 ± 0,03 **e** nenhuma repetição fora de 0,5 ± 0,10 | | alguma condição violada (investigar) |
| curva: ganho da validação de 50% para 100% dos dados (ROC-AUC macro) | < 0,0005 (platô) | 0,0005 a 0,003 | > 0,003 (ainda sobe: faltam dados) |
| **Q2 — generalizar para outro dia** | | | |
| queda validação → teste, ROC-AUC macro | < 0,005 | 0,005 a 0,02 | > 0,02 |
| queda validação → teste, F1 macro | < 0,03 | 0,03 a 0,10 | > 0,10 |
| queda validação → teste, acurácia balanceada | < 0,02 | 0,02 a 0,08 | > 0,08 |
| maior queda de recall em uma classe (classes com ao menos 200 linhas no teste) | < 0,05 | 0,05 a 0,20 | > 0,20 |
| validação adversarial: ROC-AUC treino × teste, por classe (ao menos 200 vetores únicos no teste) | < 0,6 | 0,6 a 0,8 | > 0,8 |

**Veredito de Q1** = a pior gravidade entre as duas lacunas e o embaralhamento. **Veredito de Q2** = a pior gravidade entre as cinco linhas de Q2. A curva de aprendizagem é informativa e não entra no veredito.

### Avisos sobre as regras e sobre o que o notebook não prova
- **O embaralhamento usa a MÉDIA das repetições, com tolerância fixa** (0,5 ± 0,03; e nenhuma repetição a mais de 0,10 de 0,5), e não "desvios do acaso". No binário, com 5 repetições do RF, o ROC-AUC variou de 0,43 a 0,58 em torno de 0,5 (média 0,504): um modelo treinado em rótulos embaralhados é uma **função aleatória dos atributos**, que por acaso pode se relacionar com os rótulos reais, e essa variação é muito maior que o ruído da amostra. Aquele critério gerou alerta falso. Atenção: este controle verifica se o processo, como um todo, devolve ~0,5 sem sinal; ele **não detecta** linhas repetidas entre treino e validação (um modelo treinado em rótulos embaralhados não usa os rótulos reais).
- **As regras de Q1 são as do notebook 13 do binário.** As de Q2 que usam F1 macro, acurácia balanceada e recall por classe são **convenções novas** para o multiclasse. **Eu já conhecia os resultados do teste ao escrevê-las**: o veredito de Q2 é, portanto, uma **classificação descritiva** dos sinais, não uma previsão feita antes dos dados.
- **O RF sem limite de profundidade acerta quase 100% das linhas que treinou** por construção (as folhas ficam puras). A lacuna treino − validação dele aparece como "grande" sem que isso seja, por si só, um defeito: é por isso que o veredito de Q1 também usa o embaralhamento e a curva, e que a validação cruzada (fora da dobra) é mostrada ao lado.
- **Para o SVM, o desempenho "no treino" é medido numa amostra** de `N_TREINO_SVM_AVAL` linhas do ajuste (prever o ajuste inteiro levaria ~40 min por modelo). A validação do SVM usa os escores já gravados no 11m.
- Ele **não prova** que não houve sobreajuste; mostra se **há sinais** e de que tipo. O teste continua sendo **uma única partição de outro dia**, com muitas linhas repetidas (~74%): as quedas são descritivas.
- A validação adversarial usa os **atributos** das linhas do teste (os vetores distintos guardados pelo 12m, por classe), sem calcular nenhuma métrica de modelo no teste, e **não decide nenhum hiperparâmetro**.

### Custo esperado
Da ordem de **1 a 2 horas** no seu computador (a curva e o embaralhamento do SVM e a validação adversarial pesam mais). Os parâmetros `FAZER_CURVA`, `FAZER_PERM`, `FAZER_ADV`, `CURVA_SVM`, `PERM_SVM`, `REPS_CURVA_RF`, `N_ADV` permitem encurtar. **Nada é apagado nem alterado** nas pastas dos modelos (há bloqueio por código e conferência no fim), e o servidor só é lido.

### Antes de rodar
Os notebooks 05m, 07m, 08m, 11m e 12m precisam ter terminado (o 12m guarda os vetores do teste usados na etapa 5). O `.env` precisa apontar para o servidor novo.


In [1]:
import os, re, json, time, http.client, urllib.parse
from pathlib import Path
import pandas as pd

pd.set_option("display.width", 200); pd.set_option("display.max_rows", 200); pd.set_option("display.max_colwidth", 120)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


def load_env(path: Path) -> dict:
    values = {}
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


env_path = REPO_ROOT / ".env"
if not env_path.exists():
    raise FileNotFoundError(f"Arquivo .env não encontrado em: {env_path}")
env = {**load_env(env_path), **os.environ}

CH_HOST, CH_PORT = env["CLICKHOUSE_HOST"], env["CLICKHOUSE_PORT"]
CH_USER, CH_PASSWORD, CH_DATABASE = env["CLICKHOUSE_USER"], env["CLICKHOUSE_PASSWORD"], env["CLICKHOUSE_DATABASE"]
CH_SCHEME = env.get("CLICKHOUSE_SCHEME", "https")
HTTP_TIMEOUT = 1800
print(f"ClickHouse: {CH_HOST}:{CH_PORT}/{CH_DATABASE}  |  usuário: {CH_USER}")

ClickHouse: clickhouse.afonsolelis.tech:443/railway  |  usuário: clickhouse


In [2]:
# ---------------- transporte HTTP direto (conexão reutilizada + reconexão automática só para leituras) ----------------
class ErroConexao(Exception):
    pass


CONEXAO_ESPERAS = (0, 5, 15, 45, 90)
_conn = None
ULT_STATS = None                  # estatísticas (linhas lidas etc.) da última consulta JSON


def _fechar_conexao():
    global _conn
    try:
        if _conn is not None:
            _conn.close()
    except Exception:
        pass
    _conn = None


def _http(query: str) -> pd.DataFrame:
    """Executa via HTTP. Repetir leituras é seguro; escritas não são reenviadas aqui (quem repete é carregar(), com verificação de contagem)."""
    global _conn, ULT_STATS
    params = {"database": CH_DATABASE, "default_format": "JSONCompact", "wait_end_of_query": 1}
    primeiro = re.match(r"\s*(\w+)", query)
    verbo = primeiro.group(1).upper() if primeiro else ""
    if verbo == "ALTER":
        params["mutations_sync"] = 2          # espera o DELETE terminar antes de responder
    caminho = "/?" + urllib.parse.urlencode(params)
    cab = {"X-ClickHouse-User": CH_USER, "X-ClickHouse-Key": CH_PASSWORD, "Connection": "keep-alive"}
    somente_leitura = verbo not in {"INSERT", "CREATE", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE", "DELETE", "UPDATE"}
    tentativas = CONEXAO_ESPERAS if somente_leitura else (0,)
    ultimo = None
    for espera in tentativas:
        if espera:
            print(f"  conexão instável ({type(ultimo).__name__}: {str(ultimo)[:160]}); nova tentativa em {espera}s...")
            time.sleep(espera)
        try:
            if _conn is None:
                cls = http.client.HTTPSConnection if CH_SCHEME == "https" else http.client.HTTPConnection
                _conn = cls(CH_HOST, int(CH_PORT), timeout=HTTP_TIMEOUT)
            _conn.request("POST", caminho, body=query.encode("utf-8"), headers=cab)
            resp = _conn.getresponse()
            corpo = resp.read()
        except (OSError, http.client.HTTPException) as e:
            ultimo = e
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"Falha de conexão durante escrita: {e!r}") from e
            continue
        if resp.status in (502, 503, 504) or (resp.status >= 500 and not corpo.startswith(b"Code:")):
            ultimo = Exception(f"HTTP {resp.status} [código ClickHouse: {resp.getheader('X-ClickHouse-Exception-Code')}] corpo={corpo[:300].decode('utf-8', 'replace')!r}")
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"HTTP {resp.status} durante escrita. Resposta: {corpo.decode('utf-8', 'replace')[:500]!r}")
            continue
        if resp.status != 200:
            raise Exception(corpo.decode("utf-8", "replace")[:2000])
        if not corpo.strip():
            return pd.DataFrame()
        dados = json.loads(corpo)
        ULT_STATS = dados.get("statistics")
        return pd.DataFrame(dados.get("data", []), columns=[m["name"] for m in dados.get("meta", [])])
    raise ErroConexao(f"Sem conexão com o ClickHouse após {len(tentativas)} tentativas: {ultimo!r}")

In [3]:
# ---------------- TRAVA: só é possível escrever em tabelas prep_* ----------------
PREFIXO_NOVO = "prep_"          # tabelas que ESTE notebook pode criar / alimentar / apagar

_LITERAL = re.compile(r"'[^'\\]*(?:\\.[^'\\]*)*'")      # literais entre aspas simples (aceita \' escapado)


def _sem_literais(s: str) -> str:
    # texto SQL sem o conteúdo dos literais: ';' ou 'DROP TABLE' dentro de aspas NÃO contam na checagem da trava
    return _LITERAL.sub("''", s)


_VERBOS_ESCRITA = ("CREATE", "INSERT", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE")
_VERBOS_PROIBIDOS = ("DELETE", "UPDATE", "ATTACH", "DETACH", "EXCHANGE", "GRANT", "REVOKE", "KILL", "SYSTEM", "SET", "USE")
_ALVO = re.compile(r"\b(?:TABLE|INTO|VIEW)\s+(?:IF\s+(?:NOT\s+)?EXISTS\s+)?(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)
_RENAME = re.compile(r"(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)\s+TO\s+(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)


def _tabela_de(ident: str) -> str:
    # 'banco.tabela' ou '`tabela`' -> 'tabela' (e exige que o banco, se informado, seja o atual)
    partes = [p.strip("`") for p in re.findall(r"`[^`]+`|[^.]+", ident)]
    if len(partes) == 2 and partes[0] != CH_DATABASE:
        raise PermissionError(f"Escrita bloqueada: banco '{partes[0]}' diferente do atual '{CH_DATABASE}'.")
    return partes[-1]


def validar_escrita(sql: str, permitir_prefixos=()) -> None:
    # Levanta PermissionError se o comando escreveria fora das tabelas permitidas. Leituras passam sem restrição.
    s = sql.strip().rstrip(";").strip()
    s_chk = _sem_literais(s)
    if ";" in s_chk:
        raise PermissionError("Escrita bloqueada: envie uma instrução por vez.")
    verbo = (re.match(r"\s*(\w+)", s).group(1) if re.match(r"\s*(\w+)", s) else "").upper()
    if verbo in _VERBOS_PROIBIDOS:
        raise PermissionError(f"Comando '{verbo}' não é permitido neste notebook.")
    if verbo not in _VERBOS_ESCRITA:
        return                                                    # SELECT / SHOW / DESCRIBE / EXISTS ...
    prefixos = (PREFIXO_NOVO, *permitir_prefixos)
    if verbo == "RENAME":
        nomes = [_tabela_de(x) for par in _RENAME.findall(s_chk) for x in par]
    else:
        nomes = [_tabela_de(x) for x in _ALVO.findall(s_chk)]
    if not nomes:
        raise PermissionError(f"Escrita bloqueada: não identifiquei a tabela alvo em: {s[:80]}...")
    ruins = [n for n in nomes if not n.startswith(prefixos)]
    if ruins:
        raise PermissionError(f"Escrita bloqueada em {ruins}: só é permitido escrever em tabelas com prefixo {prefixos}.")


def executar(sql: str, permitir_prefixos=()) -> pd.DataFrame:
    validar_escrita(sql, permitir_prefixos)
    return _http(sql)


consultar = executar      # leituras também passam pela trava (e ela as libera)

# autoteste da trava (não toca no servidor)
_deve_passar = ["SELECT 1", "SHOW CREATE TABLE data2_portmap", f"CREATE TABLE {PREFIXO_NOVO}x AS data2_portmap",
                f"INSERT INTO {PREFIXO_NOVO}x SELECT * FROM data2_portmap LIMIT 1", f"DROP TABLE IF EXISTS `{PREFIXO_NOVO}x`",
                f"INSERT INTO {PREFIXO_NOVO}x VALUES ('idêntica a a; teste', 'DROP TABLE data_syn')"]
_deve_bloquear = ["INSERT INTO data2_portmap SELECT 1", "DROP TABLE data2_portmap", "DROP TABLE IF EXISTS data_tftp",
                  "TRUNCATE TABLE data_syn", "ALTER TABLE data_syn DROP COLUMN label", "CREATE TABLE outra AS data_syn",
                  "RENAME TABLE data_syn TO prep_syn", "CREATE DATABASE x", "DELETE FROM data_syn WHERE 1",
                  f"INSERT INTO {PREFIXO_NOVO}x SELECT 1; DROP TABLE data_syn", "SYSTEM DROP MARK CACHE", f"INSERT INTO {PREFIXO_NOVO}x VALUES ('a'); DROP TABLE data_syn"]
for _s in _deve_passar:
    validar_escrita(_s)
for _s in _deve_bloquear:
    try:
        validar_escrita(_s); raise AssertionError(f"a trava deixou passar: {_s}")
    except PermissionError:
        pass
print(f"Trava de segurança OK: {len(_deve_passar)} comandos permitidos e {len(_deve_bloquear)} bloqueados no autoteste.")

Trava de segurança OK: 6 comandos permitidos e 12 bloqueados no autoteste.


In [4]:
# ---------- ferramentas gerais ----------
import os, sys, json, math, time, zlib, platform, threading
from pathlib import Path
from datetime import datetime
import numpy as np
try:
    import psutil
except ImportError:
    psutil = None
    print("⚠ psutil não está instalado: o pico de memória e o hardware não serão registrados. Rode `pip install psutil` e execute esta célula de novo.")
try:
    import sklearn, joblib
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
except ImportError as e:
    raise ImportError("Faltam bibliotecas: rode `pip install scikit-learn joblib psutil` e execute esta célula de novo.") from e

def lit(s): return "'" + str(s).replace("\\", "\\\\").replace("'", "\\'") + "'"
def q(c): return "`" + str(c).replace("`", "``") + "`"

def consultar_num(sql):
    df = consultar(sql)
    for c in df.columns:
        try: df[c] = pd.to_numeric(df[c])
        except (ValueError, TypeError): pass
    return df

def n_linhas(tab, onde=""):
    return int(consultar_num(f"SELECT count() AS n FROM {tab}" + (f" WHERE {onde}" if onde else "")).n[0])

def existe(tab):
    return int(consultar_num(f"SELECT count() AS n FROM system.tables WHERE database = currentDatabase() AND name = {lit(tab)}").n[0]) > 0

def colunas_view(view):
    return consultar(f"SELECT name FROM system.columns WHERE database = currentDatabase() AND `table` = {lit(view)} ORDER BY position")["name"].tolist()

def ler_view(view, onde=""):
    # Lê uma view inteira. O id_linha (UInt64) fica como texto, para não perder precisão; o resto vira número.
    cols = colunas_view(view)
    sel = ", ".join(("toString(id_linha) AS id_linha" if c == "id_linha" else q(c)) for c in cols)
    df = consultar(f"SELECT {sel} FROM {view}" + (f" WHERE {onde}" if onde else ""))
    for c in df.columns:
        if c != "id_linha": df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def executar_robusto(sql, antes=None, tentativas=5, espera=120):
    # Escritas não são repetidas pelo transporte: se faltar memória no servidor (Code 241) ou a conexão cair, esperamos e refazemos (tudo aqui é idempotente).
    for k in range(1, tentativas + 1):
        try:
            if antes: executar(antes)
            return executar(sql)
        except PermissionError:
            raise
        except Exception as e:
            if isinstance(e, ErroConexao) or "241" in str(e) or "memory limit" in str(e).lower():
                print(f"  ⚠ falha por memória/conexão (tentativa {k}/{tentativas}): {str(e)[:160]}; aguardando {espera}s e refazendo...")
                time.sleep(espera); continue
            raise
    raise RuntimeError(f"não consegui concluir após {tentativas} tentativas: {sql[:100]}...")

def distribuir(total, tamanhos, alpha, tetos):
    # Reparte `total` linhas entre os tipos (alpha = 0: partes iguais), sem passar do teto de cada um; o que sobra de um tipo pequeno vai para os demais.
    q_ = {t: 0 for t in tamanhos}; livres = [t for t in tamanhos if tetos[t] > 0]; restante = int(total)
    if sum(tetos[t] for t in livres) < restante: raise ValueError(f"não há linhas suficientes: pedido {restante:,}, disponível {sum(tetos[t] for t in livres):,}")
    while restante > 0 and livres:
        w = {t: float(tamanhos[t]) ** alpha for t in livres}; sw = sum(w.values())
        alvo = {t: restante * w[t] / sw for t in livres}
        cheios = [t for t in livres if q_[t] + alvo[t] >= tetos[t]]
        if not cheios:
            base = {t: int(alvo[t]) for t in livres}; sobra = restante - sum(base.values())
            for t in sorted(livres, key=lambda t: alvo[t] - base[t], reverse=True)[:sobra]: base[t] += 1
            for t in livres: q_[t] += base[t]
            restante = 0
        else:
            for t in cheios:
                restante -= tetos[t] - q_[t]; q_[t] = tetos[t]; livres.remove(t)
    return q_

def _limpar(o):
    if isinstance(o, dict): return {str(k): _limpar(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_limpar(v) for v in o]
    if isinstance(o, np.ndarray): return _limpar(o.tolist())
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating, float)): return None if (o != o) else float(o)
    if isinstance(o, Path): return str(o)
    return o

def salvar_json(obj, caminho):
    Path(caminho).write_text(json.dumps(_limpar(obj), indent=1, ensure_ascii=False), encoding="utf-8")

### Ferramentas
O pré-processamento, a leitura em CSV, as métricas multiclasse e as **proteções** (sobrescrita e remoção de arquivos das pastas dos modelos bloqueadas).

In [5]:
# ---------- ferramentas de métricas, pré-processamento e custo ----------
class Medidor:
    # Mede tempo de relógio, tempo de CPU (todos os núcleos do processo) e pico de RAM (RSS, amostrado a cada 50 ms) de um bloco `with`.
    def __init__(self, intervalo=0.05): self.intervalo = intervalo
    def __enter__(self):
        self.t0, self.c0 = time.perf_counter(), time.process_time(); self._ativo = psutil is not None
        if self._ativo:
            self._p = psutil.Process(); self.base = self._p.memory_info().rss; self.pico = self.base; self._parar = threading.Event()
            def amostrar():
                while not self._parar.is_set():
                    self.pico = max(self.pico, self._p.memory_info().rss); self._parar.wait(self.intervalo)
            self._th = threading.Thread(target=amostrar, daemon=True); self._th.start()
        return self
    def __exit__(self, *a):
        self.relogio, self.cpu = time.perf_counter() - self.t0, time.process_time() - self.c0
        if self._ativo:
            self._parar.set(); self._th.join(); self.pico = max(self.pico, self._p.memory_info().rss)
            self.pico_mib, self.delta_mib = self.pico / 2 ** 20, (self.pico - self.base) / 2 ** 20
        else:
            self.pico_mib = self.delta_mib = float("nan")

class PreprocessadorRF:
    # O pré-processamento do RF, igual ao da view definitiva: ±inf e vazios viram "ausente"; nas colunas com indicador, o ausente é trocado pela mediana do
    # TREINO e entra uma coluna <coluna>_ausente (0/1) no fim. A ordem das colunas do modelo é: atributos, depois os indicadores.
    def __init__(self, features, medianas, retiradas=(), versao_views=""):
        self.features, self.medianas, self.retiradas, self.versao_views = list(features), dict(medianas), list(retiradas), versao_views
        self.indicadores = [c for c in self.features if c in self.medianas]
        self.colunas_modelo = self.features + [c + "_ausente" for c in self.indicadores]
        idx = {c: i for i, c in enumerate(self.features)}
        self._pos = np.array([idx[c] for c in self.indicadores], dtype=int); self._med = np.array([self.medianas[c] for c in self.indicadores], dtype=float)
    def transformar(self, X):
        X = np.asarray(X, dtype=np.float64); X = np.where(np.isinf(X), np.nan, X)
        if len(self._pos) == 0: return X
        sub = X[:, self._pos]; aus = np.isnan(sub); X = X.copy(); X[:, self._pos] = np.where(aus, self._med, sub)
        return np.hstack([X, aus.astype(np.float64)])
    def para_dict(self):
        return {"features": self.features, "indicadores": self.indicadores, "colunas_modelo": self.colunas_modelo, "medianas_treino": self.medianas,
                "colunas_retiradas_ablacao": self.retiradas, "versao_views_rf": self.versao_views,
                "regra": "±inf e vazios = ausente; nas colunas com indicador: ausente -> mediana do treino e coluna <coluna>_ausente (0/1) no fim"}

def medir_inferencia(modelo, prep, X_modelo, X_raw, n_lat=1000, semente=42, repeticoes_lote=3):
    # Latência (1 fluxo por vez) e vazão (lote inteiro), com n_jobs = 1 e com o n_jobs original; com e sem o pré-processamento.
    rng = np.random.default_rng(semente); idx = rng.choice(len(X_modelo), size=min(n_lat, len(X_modelo)), replace=False)
    original = modelo.n_jobs; linhas = []
    for nj in dict.fromkeys([1, original]):
        modelo.set_params(n_jobs=nj)
        for i in idx[:20]: modelo.predict_proba(X_modelo[i:i + 1])                         # aquecimento
        t_mod, t_fim = [], []
        for i in idx:
            r = X_raw[i:i + 1]
            t0 = time.perf_counter(); m = prep.transformar(r); t1 = time.perf_counter(); modelo.predict_proba(m); t2 = time.perf_counter()
            t_mod.append(t2 - t1); t_fim.append(t2 - t0)
        l_mod, l_fim = np.percentile(t_mod, [50, 95, 99]) * 1000, np.percentile(t_fim, [50, 95, 99]) * 1000
        v_mod, v_fim = [], []
        for _ in range(repeticoes_lote):
            t0 = time.perf_counter(); modelo.predict_proba(X_modelo); v_mod.append(len(X_modelo) / (time.perf_counter() - t0))
            t0 = time.perf_counter(); modelo.predict_proba(prep.transformar(X_raw)); v_fim.append(len(X_raw) / (time.perf_counter() - t0))
        linhas.append({"n_jobs": nj, "n_latencia": len(idx), "latencia_modelo_p50_ms": l_mod[0], "latencia_modelo_p95_ms": l_mod[1], "latencia_modelo_p99_ms": l_mod[2],
                       "latencia_fim_a_fim_p50_ms": l_fim[0], "latencia_fim_a_fim_p95_ms": l_fim[1], "latencia_fim_a_fim_p99_ms": l_fim[2],
                       "vazao_modelo_fluxos_s": float(np.median(v_mod)), "vazao_fim_a_fim_fluxos_s": float(np.median(v_fim)), "n_lote": len(X_modelo)})
    modelo.set_params(n_jobs=original)
    return linhas

def tamanho_e_nos(modelo, caminho, compressao):
    joblib.dump(modelo, caminho, compress=compressao)
    return {"arquivo_mib": os.path.getsize(caminho) / 2 ** 20, "n_arvores": len(modelo.estimators_), "nos_total": int(sum(e.tree_.node_count for e in modelo.estimators_)),
            "profundidade_media": float(np.mean([e.get_depth() for e in modelo.estimators_])), "profundidade_max": int(max(e.get_depth() for e in modelo.estimators_))}

# ---------- métricas multiclasse (a partir da matriz de confusão; aceitam pesos por linha) ----------
def metricas_multi(y, proba, w=None):
    y = np.asarray(y, dtype=int); proba = np.asarray(proba, dtype=float); K_ = proba.shape[1]; pred = proba.argmax(axis=1); w_ = None if w is None else np.asarray(w, dtype=float)
    cm = np.zeros((K_, K_)); np.add.at(cm, (y, pred), 1.0 if w_ is None else w_)
    s, t, p, d = cm.sum(), cm.sum(axis=1), cm.sum(axis=0), np.diag(cm)                  # total, linhas por classe verdadeira, previstas por classe, acertos
    rec = np.divide(d, t, out=np.full(K_, np.nan), where=t > 0); prec = np.divide(d, p, out=np.zeros(K_), where=p > 0); f1 = np.divide(2 * d, t + p, out=np.zeros(K_), where=(t + p) > 0)
    den = math.sqrt(max((s * s - (p * p).sum()) * (s * s - (t * t).sum()), 0.0)); mcc = float((d.sum() * s - (p * t).sum()) / den) if den > 0 else 0.0
    auc_k = np.full(K_, np.nan)
    for k in range(K_):
        yk = (y == k).astype(int)
        if 0 < yk.sum() < len(yk): auc_k[k] = roc_auc_score(yk, proba[:, k], sample_weight=w_)
    esc = {"acuracia": float(d.sum() / s), "acuracia_balanceada": float(np.nanmean(rec)), "precisao_macro": float(prec.mean()), "f1_macro": float(f1.mean()), "f1_ponderado": float((f1 * t).sum() / t.sum()), "mcc": mcc, "roc_auc_ovr_macro": float(np.nanmean(auc_k))}
    return esc, {"cm": cm, "recall": rec, "precisao": prec, "f1": f1, "auc": auc_k, "suporte": t}

# autoteste das ferramentas (não toca no servidor): confere as métricas multiclasse contra o scikit-learn
from sklearn.metrics import accuracy_score as _acc, balanced_accuracy_score as _bacc, f1_score as _f1, matthews_corrcoef as _mcc, precision_score as _prec
_r = np.random.default_rng(0); _y = _r.integers(0, 4, 400); _P = _r.dirichlet(np.ones(4), 400); _P[np.arange(400), _y] += 0.35; _P /= _P.sum(axis=1, keepdims=True); _w = _r.uniform(0.5, 3, 400); _pr = _P.argmax(axis=1)
for _ww in (None, _w):
    _e, _a = metricas_multi(_y, _P, _ww)
    assert abs(_e["acuracia"] - _acc(_y, _pr, sample_weight=_ww)) < 1e-12 and abs(_e["acuracia_balanceada"] - _bacc(_y, _pr, sample_weight=_ww)) < 1e-12, _e
    assert abs(_e["f1_macro"] - _f1(_y, _pr, average="macro", sample_weight=_ww)) < 1e-12 and abs(_e["f1_ponderado"] - _f1(_y, _pr, average="weighted", sample_weight=_ww)) < 1e-12, _e
    assert abs(_e["mcc"] - _mcc(_y, _pr, sample_weight=_ww)) < 1e-12 and abs(_e["precisao_macro"] - _prec(_y, _pr, average="macro", sample_weight=_ww, zero_division=0)) < 1e-12, _e
    assert abs(_e["roc_auc_ovr_macro"] - roc_auc_score(_y, _P, multi_class="ovr", average="macro", sample_weight=_ww)) < 1e-12, _e
assert abs(metricas_multi(np.array([0, 1, 2]), np.eye(3))[0]["mcc"] - 1.0) < 1e-12
print("Autoteste das ferramentas OK | sklearn", sklearn.__version__, "| psutil", "ausente" if psutil is None else psutil.__version__)

class PreprocessadorRF:
    # O pré-processamento do RF, igual ao da view definitiva: ±inf e vazios viram "ausente"; nas colunas com indicador, o ausente é trocado pela mediana do
    # TREINO e entra uma coluna <coluna>_ausente (0/1) no fim. A ordem das colunas do modelo é: atributos, depois os indicadores.
    def __init__(self, features, medianas, retiradas=(), versao_views=""):
        self.features, self.medianas, self.retiradas, self.versao_views = list(features), dict(medianas), list(retiradas), versao_views
        self.indicadores = [c for c in self.features if c in self.medianas]
        self.colunas_modelo = self.features + [c + "_ausente" for c in self.indicadores]
        idx = {c: i for i, c in enumerate(self.features)}
        self._pos = np.array([idx[c] for c in self.indicadores], dtype=int); self._med = np.array([self.medianas[c] for c in self.indicadores], dtype=float)
    def transformar(self, X):
        X = np.asarray(X, dtype=np.float64); X = np.where(np.isinf(X), np.nan, X)
        if len(self._pos) == 0: return X
        sub = X[:, self._pos]; aus = np.isnan(sub); X = X.copy(); X[:, self._pos] = np.where(aus, self._med, sub)
        return np.hstack([X, aus.astype(np.float64)])
    def para_dict(self):
        return {"features": self.features, "indicadores": self.indicadores, "colunas_modelo": self.colunas_modelo, "medianas_treino": self.medianas,
                "colunas_retiradas_ablacao": self.retiradas, "versao_views_rf": self.versao_views,
                "regra": "±inf e vazios = ausente; nas colunas com indicador: ausente -> mediana do treino e coluna <coluna>_ausente (0/1) no fim"}

import gc

def rss_mib():
    return psutil.Process().memory_info().rss / 2 ** 20 if psutil else float("nan")

def fmt_tempo(s):
    if s != s or s == float("inf"): return "?"
    s = int(round(s)); h, r = divmod(s, 3600); m, sg = divmod(r, 60)
    return f"{h}h {m:02d}min" if h else (f"{m}min {sg:02d}s" if m else f"{sg}s")

class Barra:
    # Barra de progresso: HTML no Jupyter/VS Code; se não houver IPython, imprime uma linha a cada 5% ou 60 s.
    def __init__(self, total, desc="", intervalo=0.4):
        self.total, self.desc, self.n, self.t0, self.intervalo = max(int(total), 1), desc, 0, time.perf_counter(), intervalo
        self._ult, self._ult_pct, self._extra, self._h = 0.0, -1, "", None
        try:
            from IPython.display import display as _d, HTML as _H
            self._H = _H; self._h = _d(_H(self._html()), display_id=True)
        except Exception:
            self._h = None
    def _info(self):
        dec = time.perf_counter() - self.t0; frac = min(self.n / self.total, 1.0)
        return frac, dec, (dec / self.n * (self.total - self.n) if self.n else float("nan"))
    def _linha(self):
        frac, dec, eta = self._info()
        return f"{self.desc}: {100 * frac:5.1f}% ({self.n}/{self.total}) | decorrido {fmt_tempo(dec)} | restante ~{fmt_tempo(eta)}" + (f" | {self._extra}" if self._extra else "")
    def _html(self):
        frac, dec, eta = self._info()
        return (f"<div style='font-family:monospace;font-size:13px'><b>{self.desc}</b> <progress value='{self.n}' max='{self.total}' style='width:260px;vertical-align:middle'></progress> "
                f"<b>{100 * frac:5.1f}%</b> ({self.n}/{self.total}) &nbsp;|&nbsp; decorrido {fmt_tempo(dec)} &nbsp;|&nbsp; restante ~{fmt_tempo(eta)}"
                + (f"<br><span style='color:#555'>{self._extra}</span>" if self._extra else "") + "</div>")
    def _mostrar(self, forcar=False):
        agora = time.perf_counter()
        if self._h is not None:
            if forcar or agora - self._ult >= self.intervalo: self._h.update(self._H(self._html())); self._ult = agora
        else:
            pct = int(100 * self.n / self.total) // 5
            if forcar or pct != self._ult_pct or agora - self._ult >= 60: print(self._linha(), flush=True); self._ult_pct, self._ult = pct, agora
    def avancar(self, n=1, extra=None):
        self.n += n
        if extra is not None: self._extra = extra
        self._mostrar(forcar=self.n >= self.total)
    def fechar(self, extra=""):
        self._extra = extra or self._extra; self._mostrar(forcar=True)
        if self._h is not None: print(f"{self.desc}: concluído em {fmt_tempo(time.perf_counter() - self.t0)}")


def recall_por_tipo(y, p, g, limiar, n_tipos):
    # Posição 0 = BENIGN (aqui vale a especificidade); posições seguintes = recall de cada tipo de ataque.
    y, g = np.asarray(y), np.asarray(g); pred = np.asarray(p) >= np.float32(limiar)
    pos = np.bincount(g[y == 1], minlength=n_tipos).astype(float); det = np.bincount(g[(y == 1) & pred], minlength=n_tipos).astype(float)
    r = np.divide(det, pos, out=np.full(n_tipos, np.nan), where=pos > 0); neg = float((y == 0).sum()); r[0] = float(((y == 0) & ~pred).sum()) / neg if neg else float("nan")
    return r

class ErroMemoriaServidor(Exception):
    pass


CH_SETTINGS_LEITURA = globals().get("CH_SETTINGS_LEITURA", {"max_threads": 1, "max_block_size": 8192})      # leitura leve para o servidor (menos colunas e blocos abertos ao mesmo tempo)
ESPERAS_MEMORIA = globals().get("ESPERAS_MEMORIA", (30, 60, 120, 240, 480))                            # segundos de espera após "sem memória" (Code 241) no servidor
LIMITE_MEM_SERVIDOR_GIB = globals().get("LIMITE_MEM_SERVIDOR_GIB", 2.0)                                        # volta a tentar antes do fim da espera se a memória do servidor cair abaixo disto


def ler_csv(query, colunas, texto=()):
    # Leitura em FLUXO: o servidor devolve CSV e o pandas lê direto da conexão, sem guardar o texto inteiro nem criar milhões de objetos Python
    # (o caminho JSON do consultar() gasta várias vezes mais memória). Mesmas regras de repetição do _http: só SELECT, com novas tentativas.
    global _conn
    if not re.match(r"\s*SELECT\b", query, re.I): raise PermissionError("ler_csv só executa SELECT")
    caminho = "/?" + urllib.parse.urlencode({"database": CH_DATABASE, "default_format": "CSV", "wait_end_of_query": 1, **CH_SETTINGS_LEITURA})
    cab = {"X-ClickHouse-User": CH_USER, "X-ClickHouse-Key": CH_PASSWORD, "Connection": "keep-alive"}
    ultimo = None
    for espera in CONEXAO_ESPERAS:
        if espera:
            print(f"  conexão instável ({type(ultimo).__name__}: {str(ultimo)[:160]}); nova tentativa em {espera}s..."); time.sleep(espera)
        corpo = b""
        try:
            if _conn is None:
                cls = http.client.HTTPSConnection if CH_SCHEME == "https" else http.client.HTTPConnection
                _conn = cls(CH_HOST, int(CH_PORT), timeout=HTTP_TIMEOUT)
            _conn.request("POST", caminho, body=query.encode("utf-8"), headers=cab)
            resp = _conn.getresponse()
            if resp.status == 200:
                try:
                    return pd.read_csv(resp, header=None, names=colunas, dtype={c: (str if c in texto else np.float64) for c in colunas}, na_values=["\\N"])
                except pd.errors.EmptyDataError:
                    return pd.DataFrame({c: pd.Series(dtype=(object if c in texto else np.float64)) for c in colunas})
            corpo = resp.read()
        except (OSError, http.client.HTTPException, pd.errors.ParserError, ValueError) as e:
            ultimo = e; _fechar_conexao(); continue
        if resp.status in (502, 503, 504) or (resp.status >= 500 and not corpo.startswith(b"Code:")):
            ultimo = Exception(f"HTTP {resp.status} corpo={corpo[:300].decode('utf-8', 'replace')!r}"); _fechar_conexao(); continue
        msg = corpo.decode("utf-8", "replace")
        if "MEMORY_LIMIT_EXCEEDED" in msg or msg.startswith("Code: 241"):
            raise ErroMemoriaServidor(msg[:600])
        raise Exception(msg[:2000])
    raise ErroConexao(f"Sem conexão com o ClickHouse após {len(CONEXAO_ESPERAS)} tentativas: {ultimo!r}")


def _mem_servidor_gib():
    try:
        return float(consultar_num("SELECT toFloat64(value) AS v FROM system.asynchronous_metrics WHERE metric = 'MemoryResident'")["v"][0]) / 2 ** 30
    except Exception:
        return None


def ler_csv_com_espera(query, colunas, texto=()):
    # Se o servidor ficar sem memória (Code 241, comum num servidor de 3 GiB com merges em andamento), espera e tenta de novo; o erro é transitório.
    for k, espera in enumerate(list(ESPERAS_MEMORIA) + [None]):
        try:
            return ler_csv(query, colunas, texto)
        except ErroMemoriaServidor as e:
            if espera is None:
                raise ErroMemoriaServidor("O servidor continua sem memória depois de todas as esperas. Rode a célula de novo mais tarde: os lotes já prontos são reaproveitados. Último erro: " + str(e)[:300]) from None
            m = _mem_servidor_gib()
            print(f"  ⚠ o servidor ficou sem memória (Code 241; memória do servidor: {'?' if m is None else f'{m:.2f} GiB'}); esperando até {espera}s e tentando de novo ({k + 1}/{len(ESPERAS_MEMORIA)})...", flush=True)
            t0 = time.time()
            while time.time() - t0 < espera:
                time.sleep(min(15, espera))
                m = _mem_servidor_gib()
                if m is not None and m < LIMITE_MEM_SERVIDOR_GIB: break

_b = Barra(2, "autoteste"); _b.avancar(2); assert recall_por_tipo(np.array([0, 1, 1]), np.array([.1, .9, .2]), np.array([0, 1, 1]), 0.5, 2)[1] == 0.5
print("Autoteste das ferramentas do teste OK")

# ---------- métricas multiclasse para arrays grandes (sem cópias em float64; confusão por contagem) ----------
def metricas_multi_grande(y, P):
    y = np.asarray(y, dtype=np.int64); K_ = P.shape[1]; pred = P.argmax(axis=1)
    cm = np.bincount(y * K_ + pred, minlength=K_ * K_).reshape(K_, K_).astype(np.float64)
    s, t, p, d = cm.sum(), cm.sum(axis=1), cm.sum(axis=0), np.diag(cm)
    rec = np.divide(d, t, out=np.full(K_, np.nan), where=t > 0); prec = np.divide(d, p, out=np.zeros(K_), where=p > 0); f1 = np.divide(2 * d, t + p, out=np.zeros(K_), where=(t + p) > 0)
    den = math.sqrt(max((s * s - (p * p).sum()) * (s * s - (t * t).sum()), 0.0)); mcc = float((d.sum() * s - (p * t).sum()) / den) if den > 0 else 0.0
    auc_k = np.full(K_, np.nan)
    for k in range(K_):
        yk = (y == k)
        if 0 < yk.sum() < len(yk): auc_k[k] = roc_auc_score(yk, np.ascontiguousarray(P[:, k], dtype=np.float64))
    esc = {"acuracia": float(d.sum() / s), "acuracia_balanceada": float(np.nanmean(rec)), "precisao_macro": float(prec.mean()), "f1_macro": float(f1.mean()), "f1_ponderado": float((f1 * t).sum() / t.sum()), "mcc": mcc, "roc_auc_ovr_macro": float(np.nanmean(auc_k))}
    return esc, {"cm": cm, "recall": rec, "precisao": prec, "f1": f1, "auc": auc_k, "suporte": t, "previstas": p}

def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    ph = k / n; c = (ph + z * z / (2 * n)) / (1 + z * z / n); h = z * math.sqrt(ph * (1 - ph) / n + z * z / (4 * n * n)) / (1 + z * z / n); return (c - h, c + h)

# autoteste (não toca no servidor): confere contra o scikit-learn
from sklearn.metrics import accuracy_score as _acc, balanced_accuracy_score as _bacc, f1_score as _f1, matthews_corrcoef as _mcc
_r = np.random.default_rng(1); _y = _r.integers(0, 4, 500); _P = _r.dirichlet(np.ones(4), 500); _P[np.arange(500), _y] += 0.3; _P = (_P / _P.sum(axis=1, keepdims=True)).astype(np.float32); _pr = _P.argmax(axis=1)
_e, _a = metricas_multi_grande(_y, _P)
assert abs(_e["acuracia"] - _acc(_y, _pr)) < 1e-12 and abs(_e["acuracia_balanceada"] - _bacc(_y, _pr)) < 1e-12 and abs(_e["f1_macro"] - _f1(_y, _pr, average="macro")) < 1e-12 and abs(_e["mcc"] - _mcc(_y, _pr)) < 1e-12, _e
assert abs(_e["roc_auc_ovr_macro"] - roc_auc_score(_y, _P.astype(np.float64), multi_class="ovr", average="macro")) < 1e-9
_lo, _hi = wilson(50, 100); assert 0.40 < _lo < 0.41 and 0.59 < _hi < 0.60
print("Autoteste das ferramentas do teste multiclasse OK")

# ---------- proteções: o que este notebook NUNCA faz (sobrescrever ou apagar arquivos das pastas dos modelos) ----------
import warnings, shutil
def silenciar_aviso_sklearn():
    # Com muitos núcleos, o scikit-learn às vezes deixa a lista de filtros de aviso VAZIA (condição de corrida entre threads) e passa a emitir, a cada árvore, um aviso inofensivo
    # sobre `delayed`/`Parallel`. Este desvio descarta SÓ essa mensagem; os outros avisos continuam funcionando. Não muda nenhum resultado.
    warnings.filterwarnings("ignore", message=".*sklearn.utils.parallel.delayed.*", category=UserWarning)
    if not getattr(warnings.warn, "_filtrado_sklearn", False):
        _orig = warnings.warn
        def _warn_filtrado(message, *args, **kwargs):
            if isinstance(message, str) and message.startswith("`sklearn.utils.parallel.delayed` should be used with"): return
            return _orig(message, *args, **kwargs)
        _warn_filtrado._filtrado_sklearn = True; warnings.warn = _warn_filtrado
silenciar_aviso_sklearn()

RAIZES_TRAVADAS, CRIADOS, MANTIDOS = [], [], []
def instalar_travas(pasta, permitir_regravar=()):
    # Bloqueia gravação que SOBRESCREVA arquivo existente dentro das pastas protegidas (as dos modelos), e qualquer remoção de arquivo dentro delas. O treino NÃO é bloqueado (o diagnóstico treina modelos de teste).
    # Pode ser chamada para várias pastas; só arquivos temporários (.tmp) e acréscimos (modo "a") podem ser regravados.
    from sklearn.ensemble import RandomForestClassifier as _RF
    from sklearn.svm import SVC as _SVC
    import numpy as _np, pandas as _pd, joblib as _jl, pathlib as _pl
    RAIZES_TRAVADAS.append(str(Path(pasta).resolve())); perm = set(permitir_regravar)
    if getattr(os.remove, "_travado", False): return
    def dentro(alvo):
        try: s_ = str(Path(alvo).resolve())
        except Exception: return False
        return any(s_.startswith(r) for r in RAIZES_TRAVADAS)
    def checa(alvo, acao, permitir_acrescimo=False):
        if not isinstance(alvo, (str, os.PathLike)) or not dentro(alvo): return
        p = Path(alvo).resolve()
        if p.exists() and p.name not in perm and ".tmp" not in p.name and not permitir_acrescimo:
            raise PermissionError(f"BLOQUEADO: {acao} sobrescreveria '{p.name}', que já existe. Este notebook só cria arquivos novos.")
    def embrulha(obj, nome, pos, acao, kw="", acrescimo_ok=False):
        orig = getattr(obj, nome)
        if getattr(orig, "_travado", False): return
        def f(*a, **k):
            alvo = a[pos] if len(a) > pos else k.get(kw)
            checa(alvo, acao, permitir_acrescimo=(acrescimo_ok and k.get("mode") == "a")); return orig(*a, **k)
        f._travado = True; f.__name__ = getattr(orig, "__name__", nome); setattr(obj, nome, f)
    embrulha(_pd.DataFrame, "to_csv", 1, "gravar CSV", "path_or_buf", acrescimo_ok=True); embrulha(_np, "savez", 0, "gravar NPZ", "file"); embrulha(_np, "savez_compressed", 0, "gravar NPZ", "file"); embrulha(_np, "save", 0, "gravar NPY", "file")
    embrulha(_jl, "dump", 1, "gravar modelo", "filename"); embrulha(_pl.Path, "write_text", 0, "gravar arquivo"); embrulha(_pl.Path, "write_bytes", 0, "gravar arquivo")
    embrulha(os, "replace", 1, "substituir arquivo"); embrulha(os, "rename", 1, "renomear arquivo")
    try:
        from matplotlib.figure import Figure; embrulha(Figure, "savefig", 1, "gravar figura", "fname")
    except ImportError: pass
    def _proibe_remocao(orig):
        def f(*a, **k):
            alvo = a[0] if a else None
            if isinstance(alvo, (str, os.PathLike)) and dentro(alvo): raise PermissionError(f"BLOQUEADO: apagar '{Path(alvo).name}'. Este notebook nunca apaga arquivos.")
            return orig(*a, **k)
        f._travado = True; return f
    if not getattr(os.remove, "_travado", False): os.remove = _proibe_remocao(os.remove)
    if not getattr(_pl.Path.unlink, "_travado", False):
        _unl = _pl.Path.unlink
        def _u(self, *a, **k):
            if dentro(self): raise PermissionError(f"BLOQUEADO: apagar '{self.name}'. Este notebook nunca apaga arquivos.")
            return _unl(self, *a, **k)
        _u._travado = True; _pl.Path.unlink = _u

def gravar(caminho, escrever):
    # Cria o arquivo SÓ se ele ainda NÃO existir (nunca sobrescreve). Registra o que foi criado e o que foi mantido.
    caminho = Path(caminho); rotulo = f"{caminho.parent.name}/{caminho.name}"
    if caminho.exists(): MANTIDOS.append(rotulo); return False
    caminho.parent.mkdir(parents=True, exist_ok=True); escrever(caminho); CRIADOS.append(rotulo); return True
def gravar_csv(df, caminho, **kw): return gravar(caminho, lambda p: df.to_csv(p, **kw))
def gravar_json(obj, caminho): return gravar(caminho, lambda p: salvar_json(obj, p))
def tmp_novo(caminho): return Path(caminho).with_name(Path(caminho).stem + ".tmp" + Path(caminho).suffix)

print("Ferramentas do diagnóstico prontas")


Autoteste das ferramentas OK | sklearn 1.9.1 | psutil 7.2.2


Autoteste das ferramentas do teste OK
Autoteste das ferramentas do teste multiclasse OK
Ferramentas do diagnóstico prontas


## 1. Parâmetros, pastas, modelos e dados
Localiza os quatro modelos (treinados só no ajuste), lê o ajuste e a validação do servidor, **ativa as proteções** e **grava as regras de decisão antes de calcular qualquer resultado**.

In [6]:
# ---------- 1.1 Parâmetros, pastas, modelos e dados ----------
import joblib, zlib, hashlib
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from scipy.stats import ks_2samp
REPO = Path(globals().get("REPO_ROOT", Path.cwd())); BASE = REPO / "artefatos_modelos"
SEMENTE = globals().get("SEMENTE", 42)
FAZER_CURVA, FAZER_PERM, FAZER_ADV = globals().get("FAZER_CURVA", True), globals().get("FAZER_PERM", True), globals().get("FAZER_ADV", True)
CURVA_SVM, PERM_SVM, CURVA_PADRAO = globals().get("CURVA_SVM", True), globals().get("PERM_SVM", True), globals().get("CURVA_PADRAO", True)     # o SVM é lento: cada uma dessas partes pode ser desligada; CURVA_PADRAO inclui também os modelos padrão na curva
TAMANHOS_CURVA = tuple(globals().get("TAMANHOS_CURVA", (0.1, 0.25, 0.5, 1.0))); REPS_CURVA_RF, REPS_CURVA_SVM = globals().get("REPS_CURVA_RF", 3), globals().get("REPS_CURVA_SVM", 1)
N_TREINO_SVM_AVAL = globals().get("N_TREINO_SVM_AVAL", 20_000)          # linhas do ajuste usadas para medir o desempenho do SVM "no treino" (prever o ajuste inteiro levaria ~40 min por modelo)
N_TREINO_CURVA_SVM, N_VAL_CURVA_SVM = globals().get("N_TREINO_CURVA_SVM", 5_000), globals().get("N_VAL_CURVA_SVM", 10_000)      # amostras para avaliar cada ponto da curva do SVM
N_PERM_RF, N_PERM_SVM = globals().get("N_PERM_RF", 5), globals().get("N_PERM_SVM", 2)
N_SVM_PERM, N_VAL_PERM_SVM = globals().get("N_SVM_PERM", 7_000), globals().get("N_VAL_PERM_SVM", 20_000)                     # o SVM com rótulos embaralhados é lento: usa subamostras
MAX_ITER_PERM = globals().get("MAX_ITER_PERM", 2_000_000)               # limite de iterações do SVM no embaralhamento (sem estrutura nos rótulos ele não converge)
N_ADV, ARVORES_ADV, K_ADV = globals().get("N_ADV", 20_000), globals().get("ARVORES_ADV", 100), globals().get("K_ADV", 5)
MIN_LINHAS_ADV = globals().get("MIN_LINHAS_ADV", 100)                    # abaixo disto a classe é pulada na validação adversarial
T_SEC = {}
# regras de decisão: gravadas ANTES de calcular qualquer resultado (as mesmas da introdução)
REGRAS = {"lacuna_acc_bal": (0.01, 0.03), "lacuna_roc_auc": (0.005, 0.02), "embaralhamento_media": 0.03, "embaralhamento_max": 0.10, "sobreposicao_validacao": (0.01, 0.10), "curva_ganho_50_100": (0.0005, 0.003), "queda_roc_auc_teste": (0.005, 0.02), "queda_f1_macro_teste": (0.03, 0.10),
          "queda_acc_bal_teste": (0.02, 0.08), "queda_recall_classe": (0.05, 0.20), "min_linhas_recall_classe": 200, "auc_adversarial": (0.6, 0.8), "min_linhas_unicas_adversarial": 200}
def ultima(base, padrao, exigir):
    ps = sorted(p for p in (BASE / base).glob(padrao) if all((p / f).exists() for f in exigir)); return ps[-1] if ps else None
PASTA_RF = globals().get("PASTA_RF") or ultima("rf_multiclasse_busca", "rfm_busca_*", ("modelo_ajuste.joblib", "FINALIZADO", "metricas_teste.json"))
PASTA_RF_PAD = globals().get("PASTA_RF_PAD") or ultima("rf_multiclasse", "rfm_*", ("modelo_ajuste.joblib", "metricas_teste.json"))
PASTA_SVM = globals().get("PASTA_SVM") or ultima("svm_multiclasse_busca", "svm_busca_*", ("modelo_ajuste.joblib", "FINALIZADO"))
assert PASTA_RF and PASTA_RF_PAD and PASTA_SVM, "faltam as pastas dos modelos (notebooks 05m, 06m, 07m, 08m, 11m e 12m concluídos)"
PASTA_RF, PASTA_RF_PAD, PASTA_SVM = Path(PASTA_RF), Path(PASTA_RF_PAD), Path(PASTA_SVM)
for p_ in (PASTA_RF, PASTA_RF_PAD, PASTA_SVM): assert (p_ / "config_treino.json").exists(), f"{p_.name} sem config_treino.json"
V_AJUSTE, V_VALID, AUD_TAB, TAB_CLASSES = "prep_vw_bal_multi_treino", "prep_vw_bal_multi_validacao", "prep_views_rf_multi", "prep_classes_rf_multi"
faltam = [t for t in (V_AJUSTE, V_VALID, AUD_TAB, TAB_CLASSES) if not existe(t)]; assert not faltam, f"Rode antes os notebooks 02, 03m e 04m: faltam {faltam}"
# fotografia dos arquivos que JÁ EXISTEM nas pastas dos modelos e proteções (nada é sobrescrito nem apagado)
def foto(raiz, rot): return {(rot + "/" + str(f.relative_to(raiz))): (f.stat().st_size, f.stat().st_mtime_ns) for f in Path(raiz).rglob("*") if f.is_file()}
SNAP_ANTES = {**foto(PASTA_RF, "rf_aj"), **foto(PASTA_RF_PAD, "rf_pad"), **foto(PASTA_SVM, "svm")}
instalar_travas(PASTA_RF); instalar_travas(PASTA_RF_PAD); instalar_travas(PASTA_SVM); silenciar_aviso_sklearn()
cfg_svm, cfg_rf, cfg_rfp = (json.loads((p / "config_treino.json").read_text(encoding="utf-8")) for p in (PASTA_SVM, PASTA_RF, PASTA_RF_PAD))
cls_gravado = json.loads((PASTA_SVM / "classes.json").read_text(encoding="utf-8")); CLASSES, COD = cls_gravado["classes"], cls_gravado["codigos"]; K = len(CLASSES)
pc = json.loads((PASTA_SVM / "preprocessamento.json").read_text(encoding="utf-8")); prep = PreprocessadorRF(pc["features"], pc["medianas_treino"], pc["colunas_retiradas_ablacao"], pc["versao_views_rf"]); CM = prep.colunas_modelo
for p_ in (PASTA_RF, PASTA_RF_PAD): assert json.loads((p_ / "preprocessamento.json").read_text(encoding="utf-8"))["colunas_modelo"] == CM, f"os atributos de {p_.name} diferem dos do SVM"
assert [c for c in colunas_view(V_AJUSTE) if c not in ("id_linha", "y")] == CM == [c for c in colunas_view(V_VALID) if c not in ("id_linha", "y", "peso")], "a ordem das colunas das views difere"
def ordenar(df): return df.iloc[np.argsort(df["id_linha"].astype("uint64").to_numpy(), kind="stable")].reset_index(drop=True)
aj, va = ordenar(ler_view(V_AJUSTE)), ordenar(ler_view(V_VALID))
X_aj, y_aj, X_va, y_va = aj[CM].to_numpy(float), aj["y"].to_numpy(int), va[CM].to_numpy(float), va["y"].to_numpy(int)
for nome, X in (("ajuste", X_aj), ("validação", X_va)): assert np.isfinite(X).all(), f"valores não finitos em {nome}"
assert not set(aj["id_linha"]) & set(va["id_linha"]), "ajuste e validação compartilham linhas"

# ---- modelos (treinados só no ajuste) ----
nome_cfg = lambda C, f: f"C{C:g}_g{f:g}"; rot_svm = lambda C, f: f"C={C:g}|gamma={f:g}x"
def pontuar_rf(m, X): return m.predict_proba(X)
def pontuar_svm(m, X):
    if len(X) == 0: return np.zeros((0, K))
    return np.concatenate([m.decision_function(X[i:i + 5000]) for i in range(0, len(X), 5000)])
def cfg_rf_str(cfg): h = cfg["hiperparametros"]; return f"{h['n_estimators']}|{'None' if h['max_depth'] is None else h['max_depth']}|{h['min_samples_leaf']}"
MODELOS = {}
def registrar(chave, rotulo, familia, arquivo, pasta_grade, cfg_grade, json_teste, fits=None, descricao=""):
    m = joblib.load(arquivo)
    if familia == "RF": assert list(getattr(m, "colunas_modelo_", CM)) == CM, f"{rotulo}: as colunas do modelo diferem das das views"
    MODELOS[chave] = dict(rotulo=rotulo, familia=familia, modelo=m, pontuar=pontuar_rf if familia == "RF" else pontuar_svm, pasta_grade=pasta_grade, cfg_grade=cfg_grade, json_teste=json_teste, fits=fits, descricao=descricao)
registrar("rf_ajustado", "RF ajustado", "RF", PASTA_RF / "modelo_ajuste.joblib", PASTA_RF, cfg_rf_str(cfg_rf), PASTA_RF / "metricas_teste.json", descricao=f"{cfg_rf_str(cfg_rf)} (árvores|profundidade|folha) | {PASTA_RF.name}")
registrar("rf_padrao", "RF padrão", "RF", PASTA_RF_PAD / "modelo_ajuste.joblib", PASTA_RF, cfg_rf_str(cfg_rfp), PASTA_RF_PAD / "metricas_teste.json", descricao=f"{cfg_rf_str(cfg_rfp)} (árvores|profundidade|folha) | {PASTA_RF_PAD.name}")
hp_a, hp_p = cfg_svm["hiperparametros"], cfg_svm["padrao"]
registrar("svm_ajustado", "SVM ajustado", "SVM", PASTA_SVM / "modelo_ajuste.joblib", PASTA_SVM, rot_svm(hp_a["C"], hp_a["gamma_fator"]), PASTA_SVM / "teste_ajustado" / "metricas_teste.json", PASTA_SVM / "fits" / f"{nome_cfg(hp_a['C'], hp_a['gamma_fator'])}_completo.npz", f"C = {hp_a['C']:g} | gamma = {hp_a['gamma_fator']:g}× | {PASTA_SVM.name}")
arq_pad = PASTA_SVM / "modelos" / f"{nome_cfg(hp_p['C'], hp_p['gamma_fator'])}.joblib"
registrar("svm_padrao", "SVM padrão", "SVM", arq_pad, PASTA_SVM, rot_svm(hp_p["C"], hp_p["gamma_fator"]), PASTA_SVM / "teste_padrao" / "metricas_teste.json", PASTA_SVM / "fits" / f"{nome_cfg(hp_p['C'], hp_p['gamma_fator'])}_completo.npz", f"C = {hp_p['C']:g} | gamma = {hp_p['gamma_fator']:g}× | {PASTA_SVM.name}")
for k, M in MODELOS.items():
    assert M["json_teste"].exists(), f"{M['rotulo']}: faltam os resultados do teste ({M['json_teste']})"
    if M["fits"] is not None: assert M["fits"].exists(), f"{M['rotulo']}: faltam os escores da validação ({M['fits']})"
MAPA_DIR, UNIC_DIR = PASTA_SVM / "mapa_teste", PASTA_SVM / "vetores_unicos"; MAPA_OK = MAPA_DIR.exists() and UNIC_DIR.exists() and any(MAPA_DIR.glob("lote_*.npz")) and any(UNIC_DIR.glob("u_*.npy"))
if FAZER_ADV and not MAPA_OK: print("⚠ os vetores do teste guardados pelo 12m não foram encontrados (mapa_teste e vetores_unicos): a validação adversarial será pulada"); FAZER_ADV = False
print(f"Ajuste: {len(X_aj):,} linhas | validação: {len(X_va):,} | {K} classes: {CLASSES} | {len(CM)} atributos")
print("\nModelos (treinados só no ajuste):")
for k, M in MODELOS.items(): print(f"  {M['rotulo']:13s} {M['descricao']}")
print(f"\n🔒 Proteções: sobrescrita e remoção BLOQUEADAS nas pastas dos modelos ({len(SNAP_ANTES)} arquivos já existem e serão conferidos no final); o servidor só é lido.")
# pasta do diagnóstico e GRAVAÇÃO IMEDIATA das regras de decisão (antes de calcular qualquer resultado)
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S"); PASTA_DIAG = BASE / "diagnostico_sobreajuste_multiclasse" / f"diag_{RUN_ID}"; PASTA_DIAG.mkdir(parents=True, exist_ok=True)
salvar_json({"gravado_em": datetime.now().isoformat(timespec="seconds"), "observacao": "regras gravadas ANTES de calcular qualquer resultado deste diagnóstico (são as da introdução do notebook). As de Q2 com F1 macro, acurácia balanceada e recall por classe são convenções novas para o multiclasse, escritas já conhecendo os resultados do teste.", "regras": REGRAS}, PASTA_DIAG / "regras_decisao.json")
T_REGRAS = (PASTA_DIAG / "regras_decisao.json").stat().st_mtime_ns
print(f"\nPasta do diagnóstico: {PASTA_DIAG}")


Ajuste: 239,671 linhas | validação: 59,918 | 7 classes: ['BENIGN', 'LDAP', 'MSSQL', 'NetBIOS', 'Syn', 'UDP', 'UDPLag'] | 44 atributos

Modelos (treinados só no ajuste):
  RF ajustado   100|None|5 (árvores|profundidade|folha) | rfm_busca_20261102_001711
  RF padrão     100|None|1 (árvores|profundidade|folha) | rfm_20261101_210334
  SVM ajustado  C = 100 | gamma = 4× | svm_busca_20261107_013148
  SVM padrão    C = 1 | gamma = 1× | svm_busca_20261107_013148

🔒 Proteções: sobrescrita e remoção BLOQUEADAS nas pastas dos modelos (1286 arquivos já existem e serão conferidos no final); o servidor só é lido.

Pasta do diagnóstico: C:\Users\Bruno\downloads\tcc_codigo\railway-ch\artefatos_modelos\diagnostico_sobreajuste_multiclasse\diag_20261111_233152


## 2. Treino × validação cruzada × validação × teste
O mesmo modelo medido em quatro lugares: nas **linhas de treino** (no SVM, numa amostra), na **validação cruzada** (da busca), na **validação** do mesmo dia e no **teste** de outro dia. A **lacuna treino − validação** aponta decorar o treino (tipo 1); a **queda validação → teste** aponta ajuste ao dia (tipo 2).

In [7]:
# ---------- 2.1 Treino × CV × validação × teste ----------
t_sec0 = time.perf_counter()
def met(y, S, w=None):
    e, a = metricas_multi(y, S, w); e = dict(e); e["roc_auc"] = e.pop("roc_auc_ovr_macro"); return e, a
def basicas(y, S): e, a = met(y, S); return {"roc_auc": e["roc_auc"], "acc_bal": e["acuracia_balanceada"], "f1_macro": e["f1_macro"], "mcc": e["mcc"]}, a["recall"]
def sev(v, lims):
    if v is None or not np.isfinite(v): return "n/d"
    return "pequena" if v < lims[0] else ("moderada" if v <= lims[1] else "grande")
ORDEM_SEV = {"n/d": -1, "pequena": 0, "moderada": 1, "grande": 2}
def pior(*sevs): return max(sevs, key=lambda s: ORDEM_SEV[s]) if sevs else "n/d"
def sub_estrat(g, frac, semente, minimo=20):
    rng = np.random.default_rng(semente); idx = []
    for t in np.unique(g):
        pos = np.flatnonzero(g == t); n = len(pos) if frac >= 1 else min(len(pos), max(int(np.ceil(frac * len(pos))), minimo))
        idx.append(rng.choice(pos, size=n, replace=False) if n < len(pos) else pos)
    return np.sort(np.concatenate(idx))
def cv_de(M):
    nan = {"roc_auc": np.nan, "acc_bal": np.nan, "f1_macro": np.nan, "mcc": np.nan}
    try:
        g = pd.read_csv(M["pasta_grade"] / "resultados_grade.csv", keep_default_na=False); chave = g["config"].astype(str).str.replace(r"\.0(\||$)", r"\1", regex=True); g = g[chave == M["cfg_grade"]].iloc[0]       # "100|None|5" (aceita também "100.0|None|5")
        return {"roc_auc": float(g["cv_roc_auc"]), "acc_bal": float(g["cv_acuracia_balanceada"]), "f1_macro": float(g["cv_f1_macro"]), "mcc": float(g["cv_mcc"])}
    except Exception as e: print(f"⚠ CV de {M['rotulo']} não lida ({type(e).__name__}: {e}); coluna fica vazia"); return nan
def teste_de(M):
    j = json.loads(M["json_teste"].read_text(encoding="utf-8")); m = j["metricas"]; pc_ = j["por_classe"]
    return {"roc_auc": m["roc_auc_ovr_macro"], "acc_bal": m["acuracia_balanceada"], "f1_macro": m["f1_macro"], "mcc": m["mcc"]}, {c: pc_[c]["recall"] for c in CLASSES}, {c: int(pc_[c]["linhas no teste"]) for c in CLASSES}
IDX_TREINO_SVM = sub_estrat(y_aj, min(1.0, N_TREINO_SVM_AVAL / len(y_aj)), SEMENTE)            # as mesmas linhas para os dois SVM
ESC, linhas, REC_VAL, REC_TE, N_TE = {}, [], {}, {}, {}
barra = Barra(len(MODELOS), "Pontuando ajuste e validação")
for chave, M in MODELOS.items():
    t0 = time.perf_counter(); eh_rf = M["familia"] == "RF"
    if eh_rf: idx_tr = np.arange(len(y_aj)); s_aj, s_va = M["pontuar"](M["modelo"], X_aj), M["pontuar"](M["modelo"], X_va)
    else: idx_tr = IDX_TREINO_SVM; s_aj = M["pontuar"](M["modelo"], X_aj[idx_tr]); s_va = np.load(M["fits"])["scores"].astype(np.float64)                  # escores da validação gravados pelo 11m
    assert np.isfinite(s_aj).all() and np.isfinite(s_va).all() and s_va.shape == (len(y_va), K), f"escores inválidos em {M['rotulo']}"
    ESC[chave] = {"aj": s_aj, "idx_aj": idx_tr, "va": s_va}
    tr, rec_tr = basicas(y_aj[idx_tr], s_aj); vv, rec_v = basicas(y_va, s_va); cv = cv_de(M); te, rec_t, n_t = teste_de(M); REC_VAL[chave], REC_TE[chave], N_TE[chave] = dict(zip(CLASSES, rec_v)), rec_t, n_t
    r = {"modelo": M["rotulo"], "linhas_treino_avaliadas": len(idx_tr)}
    for pref, d in (("treino", tr), ("cv", cv), ("validacao", vv), ("teste", te)):
        for k, v in d.items(): r[f"{pref}_{k}"] = v
    r["lacuna_acc_bal"] = tr["acc_bal"] - vv["acc_bal"]; r["lacuna_roc_auc"] = tr["roc_auc"] - vv["roc_auc"]
    r["queda_roc_auc_teste"] = vv["roc_auc"] - te["roc_auc"]; r["queda_f1_macro_teste"] = vv["f1_macro"] - te["f1_macro"]; r["queda_acc_bal_teste"] = vv["acc_bal"] - te["acc_bal"]
    quedas = {c: REC_VAL[chave][c] - rec_t[c] for c in CLASSES if n_t[c] >= REGRAS["min_linhas_recall_classe"]}; c_pior = max(quedas, key=quedas.get); r["queda_recall_classe"] = quedas[c_pior]; r["classe_maior_queda_recall"] = c_pior
    linhas.append(r); barra.avancar(1, f"{M['rotulo']}: {time.perf_counter() - t0:.0f} s")
barra.fechar()
TAB = pd.DataFrame(linhas).set_index("modelo")
print("\n== Métricas por origem (previsão = maior probabilidade no RF e maior escore no SVM; o SVM 'no treino' é medido numa amostra de " + f"{len(IDX_TREINO_SVM):,} linhas do ajuste) ==")
display(TAB[[c for c in TAB.columns if c.split("_")[0] in ("treino", "cv", "validacao", "teste")]].round(5))
print("\n== Lacunas e quedas ==")
LAC = TAB[["lacuna_acc_bal", "lacuna_roc_auc", "queda_roc_auc_teste", "queda_f1_macro_teste", "queda_acc_bal_teste", "queda_recall_classe", "classe_maior_queda_recall"]].copy()
LAC["Q1 acc.bal."] = [sev(max(v, 0), REGRAS["lacuna_acc_bal"]) for v in TAB.lacuna_acc_bal]; LAC["Q1 ROC-AUC"] = [sev(max(v, 0), REGRAS["lacuna_roc_auc"]) for v in TAB.lacuna_roc_auc]
LAC["Q2 ROC-AUC"] = [sev(v, REGRAS["queda_roc_auc_teste"]) for v in TAB.queda_roc_auc_teste]; LAC["Q2 F1 macro"] = [sev(v, REGRAS["queda_f1_macro_teste"]) for v in TAB.queda_f1_macro_teste]
LAC["Q2 acc.bal."] = [sev(v, REGRAS["queda_acc_bal_teste"]) for v in TAB.queda_acc_bal_teste]; LAC["Q2 recall classe"] = [sev(max(v, 0), REGRAS["queda_recall_classe"]) for v in TAB.queda_recall_classe]
display(LAC.round(5))
print("\n== Recall por classe: validação → teste ==")
rec_tab = pd.DataFrame({**{f"{MODELOS[k]['rotulo']} (validação)": REC_VAL[k] for k in MODELOS}, **{f"{MODELOS[k]['rotulo']} (teste)": REC_TE[k] for k in MODELOS}}); display(rec_tab.round(4))
T_SEC["2.1_tabela"] = time.perf_counter() - t_sec0


Pontuando ajuste e validação: concluído em 7min 04s

== Métricas por origem (previsão = maior probabilidade no RF e maior escore no SVM; o SVM 'no treino' é medido numa amostra de 20,002 linhas do ajuste) ==


,treino_roc_auc,treino_acc_bal,treino_f1_macro,treino_mcc,cv_roc_auc,cv_acc_bal,cv_f1_macro,cv_mcc,validacao_roc_auc,validacao_acc_bal,validacao_f1_macro,validacao_mcc,teste_roc_auc,teste_acc_bal,teste_f1_macro,teste_mcc
modelo,,,,,,,,,,,,,,,,
RF ajustado,0.99780,0.94240,0.93639,0.93281,0.99489,0.92297,0.91773,0.91411,0.99492,0.92246,0.91741,0.91361,0.92483,0.67714,0.41477,0.36879
RF padrão,0.99991,0.99107,0.98670,0.98814,0.99047,0.90407,0.90231,0.89874,0.99020,0.90255,0.90069,0.89689,0.91328,0.69079,0.56262,0.43242
SVM ajustado,0.97352,0.86966,0.87491,0.88233,0.97334,0.86908,0.87373,0.88105,0.97355,0.87104,0.87566,0.88245,0.89974,0.73350,0.60222,0.51724
SVM padrão,0.97508,0.86120,0.86486,0.87179,0.97523,0.86085,0.86475,0.87135,0.97572,0.86296,0.86621,0.87212,0.88547,0.69141,0.61190,0.53467



== Lacunas e quedas ==


,lacuna_acc_bal,lacuna_roc_auc,queda_roc_auc_teste,queda_f1_macro_teste,queda_acc_bal_teste,queda_recall_classe,classe_maior_queda_recall,Q1 acc.bal.,Q1 ROC-AUC,Q2 ROC-AUC,Q2 F1 macro,Q2 acc.bal.,Q2 recall classe
modelo,,,,,,,,,,,,,
RF ajustado,0.01994,0.00288,0.07010,0.50264,0.24532,0.81732,Syn,moderada,pequena,grande,grande,grande,grande
RF padrão,0.08852,0.00971,0.07692,0.33807,0.21176,0.71046,Syn,grande,moderada,grande,grande,grande,grande
SVM ajustado,-0.00138,-0.00003,0.07382,0.27344,0.13754,0.54310,MSSQL,pequena,pequena,grande,grande,grande,grande
SVM padrão,-0.00176,-0.00064,0.09025,0.25431,0.17155,0.70330,UDPLag,pequena,pequena,grande,grande,grande,grande



== Recall por classe: validação → teste ==


,RF ajustado (validação),RF padrão (validação),SVM ajustado (validação),SVM padrão (validação),RF ajustado (teste),RF padrão (teste),SVM ajustado (teste),SVM padrão (teste)
BENIGN,0.9993,0.9990,0.9983,0.9959,0.9806,0.9824,0.9850,0.9884
LDAP,0.9784,0.9646,0.9560,0.9507,0.9926,0.9632,0.9823,0.9847
MSSQL,0.8884,0.8804,0.9179,0.8984,0.3411,0.3681,0.3748,0.3835
NetBIOS,0.8458,0.7630,0.5441,0.5380,0.9987,0.9990,0.9987,0.9964
Syn,0.9790,0.9550,0.9912,0.9729,0.1617,0.2445,0.4736,0.5273
UDP,0.9409,0.9180,0.9884,0.9816,0.9081,0.8744,0.9736,0.9596
UDPLag,0.8254,0.8379,0.7014,0.7033,0.3573,0.4039,0.3465,0.0000


## 3. Curva de aprendizagem
Treina com 10%, 25%, 50% e 100% do ajuste e mede treino e validação em cada tamanho. Se a validação ainda sobe de 50% para 100%, faltam dados; se a lacuna se fecha, o modelo não está decorando. O ponto de 100% usa o modelo já treinado, sem retreinar.

In [8]:
# ---------- 3.1 Curva de aprendizagem ----------
t_sec0 = time.perf_counter()
CURVA, CURVA_RES, GANHO_50_100, FIG_CURVA = pd.DataFrame(), pd.DataFrame(), {}, None
if FAZER_CURVA:
    quem = [k for k in MODELOS if (k.endswith("ajustado") or CURVA_PADRAO) and (MODELOS[k]["familia"] == "RF" or CURVA_SVM)]
    IDX_VAL_SVM = sub_estrat(y_va, min(1.0, N_VAL_CURVA_SVM / len(y_va)), SEMENTE); linhas = []
    n_fits = sum((1 if f >= 1 else (REPS_CURVA_RF if MODELOS[k]["familia"] == "RF" else REPS_CURVA_SVM)) for k in quem for f in TAMANHOS_CURVA)
    barra = Barra(n_fits, "Curva de aprendizagem")
    for k in quem:
        M = MODELOS[k]; eh_rf = M["familia"] == "RF"
        for frac in TAMANHOS_CURVA:
            for r in range(1 if frac >= 1 else (REPS_CURVA_RF if eh_rf else REPS_CURVA_SVM)):
                idx = sub_estrat(y_aj, frac, SEMENTE + 1000 * r + int(round(frac * 100))); rng = np.random.default_rng(SEMENTE + 5 * r + int(round(frac * 100)))
                t0 = time.perf_counter()
                if frac >= 1: est, t_fit = M["modelo"], float("nan")                                          # 100%: o modelo já treinado (o mesmo da busca), sem retreinar
                else: est = clone(M["modelo"]).fit(X_aj[idx], y_aj[idx]); t_fit = time.perf_counter() - t0
                idx_tr_av = idx if eh_rf else np.sort(rng.choice(idx, size=min(N_TREINO_CURVA_SVM, len(idx)), replace=False))
                a, _ = basicas(y_aj[idx_tr_av], M["pontuar"](est, X_aj[idx_tr_av]))
                if eh_rf: b, _ = basicas(y_va, M["pontuar"](est, X_va)); n_av = len(y_va)
                else:
                    s_v = ESC[k]["va"][IDX_VAL_SVM] if frac >= 1 else M["pontuar"](est, X_va[IDX_VAL_SVM]); b, _ = basicas(y_va[IDX_VAL_SVM], s_v); n_av = len(IDX_VAL_SVM)
                linhas.append({"modelo": M["rotulo"], "fracao": frac, "sorteio": r, "n_linhas": len(idx), "n_avaliacao_validacao": n_av, "treino_roc_auc": a["roc_auc"], "treino_acc_bal": a["acc_bal"], "treino_f1_macro": a["f1_macro"],
                               "validacao_roc_auc": b["roc_auc"], "validacao_acc_bal": b["acc_bal"], "validacao_f1_macro": b["f1_macro"], "validacao_mcc": b["mcc"], "tempo_treino_s": t_fit})
                barra.avancar(1, f"{M['rotulo']} | {int(100 * frac)}% ({len(idx):,} linhas) | " + (f"treino {t_fit:.0f} s" if frac < 1 else "modelo já treinado") + f" | {time.perf_counter() - t0:.0f} s no total")
    barra.fechar(); CURVA = pd.DataFrame(linhas)
    CURVA_RES = CURVA.groupby(["modelo", "fracao"]).agg(n_linhas=("n_linhas", "first"), treino_roc_auc=("treino_roc_auc", "mean"), validacao_roc_auc=("validacao_roc_auc", "mean"), validacao_roc_auc_dp=("validacao_roc_auc", "std"),
                                                         treino_acc_bal=("treino_acc_bal", "mean"), validacao_acc_bal=("validacao_acc_bal", "mean"), validacao_f1_macro=("validacao_f1_macro", "mean"), validacao_mcc=("validacao_mcc", "mean"), tempo_treino_s=("tempo_treino_s", "mean")).reset_index()
    CURVA_RES["lacuna_acc_bal"] = CURVA_RES.treino_acc_bal - CURVA_RES.validacao_acc_bal; CURVA_RES["lacuna_roc_auc"] = CURVA_RES.treino_roc_auc - CURVA_RES.validacao_roc_auc
    print("\n== Curva de aprendizagem (média dos sorteios; dp = desvio da validação entre sorteios; no SVM, treino e validação são medidos em amostras) =="); display(CURVA_RES.round(5))
    for rot, d in CURVA_RES.groupby("modelo"):
        d = d.set_index("fracao")
        if 0.5 in d.index and 1.0 in d.index:
            GANHO_50_100[rot] = float(d.loc[1.0, "validacao_roc_auc"] - d.loc[0.5, "validacao_roc_auc"])
            print(f"  {rot}: ganho da validação de 50% para 100% dos dados = {GANHO_50_100[rot]:+.5f} ROC-AUC → {sev(GANHO_50_100[rot], REGRAS['curva_ganho_50_100']).replace('pequena', 'platô').replace('moderada', 'sobe pouco').replace('grande', 'ainda sobe')}")
    try:
        import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
        rots = list(CURVA_RES.modelo.unique()); FIG_CURVA, ax = plt.subplots(len(rots), 2, figsize=(11, 3.6 * len(rots)), squeeze=False)
        for i, rot in enumerate(rots):
            d = CURVA_RES[CURVA_RES.modelo == rot]
            for j, (c_tr, c_va, tit) in enumerate((("treino_roc_auc", "validacao_roc_auc", "ROC-AUC macro"), ("treino_acc_bal", "validacao_acc_bal", "acurácia balanceada"))):
                a_ = ax[i, j]; a_.plot(d.n_linhas, d[c_tr], "o--", label="treino (linhas usadas)"); a_.plot(d.n_linhas, d[c_va], "o-", label="validação")
                if j == 0 and d.validacao_roc_auc_dp.notna().any(): a_.errorbar(d.n_linhas, d[c_va], yerr=d.validacao_roc_auc_dp.fillna(0), fmt="none", capsize=3, color="C1")
                a_.set_title(f"{rot}: {tit}"); a_.set_xlabel("linhas de treino"); a_.grid(alpha=0.3)
                if i == 0 and j == 0: a_.legend()
        FIG_CURVA.suptitle("Curva de aprendizagem: treino × validação"); FIG_CURVA.tight_layout(); plt.close(FIG_CURVA)
    except ImportError: print("(matplotlib não está instalado: gráfico não gerado)")
else: print("Curva de aprendizagem desligada (FAZER_CURVA = False).")
T_SEC["3.1_curva"] = time.perf_counter() - t_sec0


Curva de aprendizagem: concluído em 27min 05s

== Curva de aprendizagem (média dos sorteios; dp = desvio da validação entre sorteios; no SVM, treino e validação são medidos em amostras) ==


,modelo,fracao,n_linhas,treino_roc_auc,validacao_roc_auc,validacao_roc_auc_dp,treino_acc_bal,validacao_acc_bal,validacao_f1_macro,validacao_mcc,tempo_treino_s,lacuna_acc_bal,lacuna_roc_auc
0,RF ajustado,0.10,23968,0.99742,0.99309,0.00017,0.93320,0.91118,0.90586,0.90290,0.72192,0.02202,0.00433
1,RF ajustado,0.25,59918,0.99766,0.99419,0.00011,0.93870,0.91683,0.91170,0.90828,2.81960,0.02187,0.00347
2,RF ajustado,0.50,119836,0.99775,0.99455,0.00009,0.94089,0.92036,0.91520,0.91171,10.88845,0.02053,0.00320
3,RF ajustado,1.00,239671,0.99780,0.99492,NaN,0.94240,0.92246,0.91741,0.91361,NaN,0.01994,0.00288
4,RF padrão,0.10,23968,1.00000,0.98959,0.00047,0.99857,0.89969,0.89788,0.89425,0.90614,0.09888,0.01041
5,RF padrão,0.25,59918,0.99999,0.99056,0.00039,0.99726,0.90429,0.90215,0.89812,3.21076,0.09297,0.00943
6,RF padrão,0.50,119836,0.99997,0.99045,0.00042,0.99525,0.90520,0.90319,0.89891,11.90406,0.09005,0.00952
7,RF padrão,1.00,239671,0.99991,0.99020,NaN,0.99107,0.90255,0.90069,0.89689,NaN,0.08852,0.00971
8,SVM ajustado,0.10,23968,0.97754,0.97693,NaN,0.86667,0.86821,0.87545,0.88284,11.16041,-0.00154,0.00060
9,SVM ajustado,0.25,59918,0.97348,0.97382,NaN,0.86623,0.86909,0.87124,0.87679,112.44305,-0.00286,-0.00033


  RF ajustado: ganho da validação de 50% para 100% dos dados = +0.00037 ROC-AUC → platô
  RF padrão: ganho da validação de 50% para 100% dos dados = -0.00025 ROC-AUC → platô
  SVM ajustado: ganho da validação de 50% para 100% dos dados = -0.00020 ROC-AUC → platô
  SVM padrão: ganho da validação de 50% para 100% dos dados = +0.00013 ROC-AUC → platô


## 4. Embaralhamento de rótulos (controle negativo)
Treina com os rótulos embaralhados e mede na validação real: sem relação entre atributos e rótulo, o ROC-AUC macro deve ficar em ~0,5. Se ficar longe de 0,5, há vazamento ou erro no processo.

In [9]:
# ---------- 4.1 Embaralhamento de rótulos ----------
import warnings
t_sec0 = time.perf_counter(); PERM, PERM_RES = pd.DataFrame(), pd.DataFrame()
if FAZER_PERM:
    quem = [k for k in MODELOS if k.endswith("ajustado") and (MODELOS[k]["familia"] == "RF" or PERM_SVM)]; linhas = []
    barra = Barra(sum(N_PERM_RF if MODELOS[k]["familia"] == "RF" else N_PERM_SVM for k in quem), "Embaralhamento de rótulos")
    for k in quem:
        M = MODELOS[k]; eh_rf = M["familia"] == "RF"; n_rep = N_PERM_RF if eh_rf else N_PERM_SVM
        if eh_rf: idx_fit, idx_ev = np.arange(len(y_aj)), np.arange(len(y_va))
        else: idx_fit, idx_ev = sub_estrat(y_aj, min(1.0, N_SVM_PERM / len(y_aj)), SEMENTE), sub_estrat(y_va, min(1.0, N_VAL_PERM_SVM / len(y_va)), SEMENTE)
        for r in range(n_rep):
            y_emb = np.random.default_rng(SEMENTE + 7 * r + 1).permutation(y_aj[idx_fit]); est = clone(M["modelo"])
            if not eh_rf: est.set_params(svm__max_iter=MAX_ITER_PERM)
            t0 = time.perf_counter()
            with warnings.catch_warnings(): warnings.simplefilter("ignore"); est.fit(X_aj[idx_fit], y_emb)
            b, _ = basicas(y_va[idx_ev], M["pontuar"](est, X_va[idx_ev]))
            linhas.append({"modelo": M["rotulo"], "repeticao": r, "n_treino": len(idx_fit), "n_avaliacao": len(idx_ev), "roc_auc_embaralhado": b["roc_auc"], "desvio_de_0_5": b["roc_auc"] - 0.5, "acuracia_balanceada": b["acc_bal"], "acaso_acc_bal": 1.0 / K})
            barra.avancar(1, f"{M['rotulo']} | repetição {r + 1}/{n_rep} | ROC-AUC {b['roc_auc']:.4f} | {time.perf_counter() - t0:.0f} s")
    barra.fechar(); PERM = pd.DataFrame(linhas); PERM["desvio_abs"] = PERM.desvio_de_0_5.abs()
    PERM_RES = PERM.groupby("modelo").agg(repeticoes=("roc_auc_embaralhado", "size"), roc_auc_medio=("roc_auc_embaralhado", "mean"), roc_auc_min=("roc_auc_embaralhado", "min"), roc_auc_max=("roc_auc_embaralhado", "max"), max_desvio_abs=("desvio_abs", "max"), acc_bal_media=("acuracia_balanceada", "mean")).reset_index()
    PERM_RES["situacao"] = ["ok (média a menos de %g e cada repetição a menos de %g de 0,5; a variação entre repetições é esperada)" % (REGRAS["embaralhamento_media"], REGRAS["embaralhamento_max"]) if (abs(m_ - 0.5) <= REGRAS["embaralhamento_media"] and d <= REGRAS["embaralhamento_max"]) else "ALERTA: longe de 0,5" for m_, d in zip(PERM_RES.roc_auc_medio, PERM_RES.max_desvio_abs)]
    print("\n== ROC-AUC macro na validação real, com o modelo treinado em rótulos embaralhados (esperado ~0,5; a acurácia balanceada esperada ao acaso é " + f"{1 / K:.3f}) =="); display(PERM_RES.round(4))
else: print("Embaralhamento desligado (FAZER_PERM = False).")
T_SEC["4.1_embaralhamento"] = time.perf_counter() - t_sec0


Embaralhamento de rótulos: concluído em 11min 36s

== ROC-AUC macro na validação real, com o modelo treinado em rótulos embaralhados (esperado ~0,5; a acurácia balanceada esperada ao acaso é 0.143) ==


,modelo,repeticoes,roc_auc_medio,roc_auc_min,roc_auc_max,max_desvio_abs,acc_bal_media,situacao
0,RF ajustado,5,0.4972,0.4895,0.5028,0.0105,0.1452,"ok (média a menos de 0.03 e cada repetição a menos de 0.1 de 0,5; a variação entre repetições é esperada)"
1,SVM ajustado,2,0.4695,0.4161,0.5228,0.0839,0.1954,"ALERTA: longe de 0,5"


## 4.2 Linhas repetidas entre o ajuste e a validação
O embaralhamento (4) **não** detecta linhas repetidas entre o ajuste e a validação. Esta etapa conta, **diretamente e sem treinar nada**, as linhas da validação cujo vetor de atributos é **idêntico** ao de alguma linha do ajuste, e quantas delas têm rótulo **diferente** (conflito). Se a fração for relevante, a validação do mesmo dia é **otimista**, e a lacuna treino − validação subestima o sobreajuste.

In [10]:
# ---------- 4.2 Linhas repetidas entre o ajuste e a validação ----------
# O embaralhamento de rótulos (4.1) NÃO detecta linhas repetidas entre o ajuste e a validação (um modelo treinado em rótulos embaralhados não usa os rótulos reais). Esta contagem direta detecta: ela
# compara os vetores de atributos (exatamente, byte a byte) e mostra quantas linhas da validação têm um vetor idêntico a uma linha do ajuste, e quantas dessas têm RÓTULO diferente (conflito).
t_sec0 = time.perf_counter()
def _chaves(X):
    X = np.ascontiguousarray(X, dtype=np.float64)
    if X.shape[0] == 0: return []
    mv = memoryview(X).cast("B"); w = X.shape[1] * 8; return [hashlib.blake2b(mv[i * w:(i + 1) * w], digest_size=16).digest() for i in range(X.shape[0])]       # chave de 128 bits do vetor
rot_aj = {}
for k_, y_ in zip(_chaves(X_aj), y_aj): rot_aj.setdefault(k_, set()).add(int(y_))
ch_va = _chaves(X_va); dup = np.array([k_ in rot_aj for k_ in ch_va], bool); conf = np.array([(k_ in rot_aj) and (int(y_) not in rot_aj[k_]) for k_, y_ in zip(ch_va, y_va)], bool)
NOMES_OV, GVA_OV = CLASSES, y_va
linhas = [{"grupo": "todas", "linhas na validação": len(y_va), "com vetor idêntico ao de uma linha do ajuste": int(dup.sum()), "fração": float(dup.mean()), "idênticas com rótulo DIFERENTE (conflito)": int(conf.sum())}]
for gi, nome in enumerate(NOMES_OV):
    m = GVA_OV == gi
    if m.any(): linhas.append({"grupo": nome, "linhas na validação": int(m.sum()), "com vetor idêntico ao de uma linha do ajuste": int(dup[m].sum()), "fração": float(dup[m].mean()), "idênticas com rótulo DIFERENTE (conflito)": int(conf[m].sum())})
OVERLAP_TAB = pd.DataFrame(linhas).set_index("grupo"); OV_FRAC = float(dup.mean()); OV_SEV = sev(OV_FRAC, REGRAS["sobreposicao_validacao"])
OV_SEV_GRUPO = pior(*[sev(r["fração"], REGRAS["sobreposicao_validacao"]) for g_, r in OVERLAP_TAB.iterrows() if g_ != "todas" and r["linhas na validação"] >= 100])
OV_INFO = {"fracao_validacao_com_vetor_do_ajuste": OV_FRAC, "linhas": int(dup.sum()), "conflitos": int(conf.sum()), "gravidade_geral": OV_SEV, "pior_gravidade_por_grupo": OV_SEV_GRUPO, "regra": REGRAS["sobreposicao_validacao"]}
print("== Linhas da validação cujo vetor de atributos é IDÊNTICO ao de alguma linha do ajuste ==")
display(OVERLAP_TAB.round(4))
print(f"\nNo total, {100 * OV_FRAC:.2f}% das linhas da validação repetem um vetor do ajuste ({int(dup.sum()):,} de {len(y_va):,}); {int(conf.sum()):,} delas têm rótulo diferente. Gravidade geral: {OV_SEV}; pior grupo (com ao menos 100 linhas): {OV_SEV_GRUPO}.")
print("Se for relevante, a validação do mesmo dia é OTIMISTA (o modelo já viu aquele vetor), e a lacuna treino − validação subestima o sobreajuste." if OV_SEV in ("moderada", "grande") else "A validação é praticamente independente do ajuste quanto a vetores repetidos.")
T_SEC["4.2_sobreposicao"] = time.perf_counter() - t_sec0


== Linhas da validação cujo vetor de atributos é IDÊNTICO ao de alguma linha do ajuste ==


,linhas na validação,com vetor idêntico ao de uma linha do ajuste,fração,idênticas com rótulo DIFERENTE (conflito)
grupo,,,,
todas,59918,1207,0.0201,1129
BENIGN,10000,10,0.0010,1
LDAP,6020,98,0.0163,98
MSSQL,10000,444,0.0444,440
NetBIOS,3898,399,0.1024,362
Syn,10000,77,0.0077,66
UDP,10000,44,0.0044,44
UDPLag,10000,135,0.0135,118



No total, 2.01% das linhas da validação repetem um vetor do ajuste (1,207 de 59,918); 1,129 delas têm rótulo diferente. Gravidade geral: moderada; pior grupo (com ao menos 100 linhas): grande.
Se for relevante, a validação do mesmo dia é OTIMISTA (o modelo já viu aquele vetor), e a lacuna treino − validação subestima o sobreajuste.


## 5. Validação adversarial, classe a classe (os dois dias são diferentes?)
Para cada classe, um classificador tenta distinguir **linhas do dia do treino** de **vetores distintos do dia do teste** (os guardados pelo 12m). ROC-AUC ~0,5 = os dois dias parecem iguais; ~1 = são diferentes. O **controle** (duas metades do treino) deve dar ~0,5. A tabela mostra os **atributos que mais mudam** (importância, KS e percentis), junto com o recall de cada modelo na validação e no teste.

In [11]:
# ---------- 5.1 Validação adversarial por classe ----------
t_sec0 = time.perf_counter(); ADV, ADV_FEAT, FIG_ADV = pd.DataFrame(), {}, None
if FAZER_ADV:
    arqs_mapa = sorted(MAPA_DIR.glob("lote_*_de_*.npz")); y_te_all = np.concatenate([np.load(f)["y"] for f in arqs_mapa]); uid_te_all = np.concatenate([np.load(f)["uid"] for f in arqs_mapa])
    arqs_u = sorted(UNIC_DIR.glob("u_*.npy")); n_por_u = [len(np.load(f, mmap_mode="r")) for f in arqs_u]; inicio_u = np.concatenate([[0], np.cumsum(n_por_u)]).astype(np.int64)
    assert int(uid_te_all.max()) + 1 == int(inicio_u[-1]), "o mapa do teste e os vetores distintos do 12m não fecham"
    def vetores_do_teste(uids):
        # os vetores distintos (float64, os mesmos do 12m) de uma lista de ids, lidos por memória mapeada
        uids = np.sort(uids); out = np.empty((len(uids), len(CM)))
        for j in range(len(arqs_u)):
            m = (uids >= inicio_u[j]) & (uids < inicio_u[j + 1])
            if m.any(): out[m] = np.load(arqs_u[j], mmap_mode="r")[uids[m] - inicio_u[j]]
        return out
    def rf_dom(semente): return RandomForestClassifier(ARVORES_ADV, max_depth=12, min_samples_leaf=5, n_jobs=-1, random_state=semente)
    def auc_dominio(A, B, semente):
        X = np.vstack([A, B]); d = np.r_[np.zeros(len(A), int), np.ones(len(B), int)]; oof = np.zeros(len(d))
        for itr, ite in StratifiedKFold(K_ADV, shuffle=True, random_state=semente).split(X, d): oof[ite] = rf_dom(semente).fit(X[itr], d[itr]).predict_proba(X[ite])[:, 1]
        return float(roc_auc_score(d, oof)), X, d
    barra = Barra(K, "Validação adversarial"); linhas = []
    for c in CLASSES:
        t0 = time.perf_counter(); k = COD[c]
        tr_c = pd.DataFrame(np.vstack([X_aj[y_aj == k], X_va[y_va == k]])).drop_duplicates(ignore_index=True).sample(frac=1, random_state=SEMENTE).reset_index(drop=True)            # linhas únicas do dia do treino (ajuste + validação)
        uids_c = np.unique(uid_te_all[y_te_all == k]); n_unicas_te = len(uids_c); n_linhas_te = int((y_te_all == k).sum())
        sel = np.random.default_rng(SEMENTE).choice(uids_c, size=min(N_ADV, len(uids_c)), replace=False) if len(uids_c) else np.array([], np.int64); d_te = vetores_do_teste(sel) if len(sel) else np.zeros((0, len(CM)))
        d_tr = tr_c.head(2 * N_ADV); m = min(len(d_tr), len(d_te))
        linha = {"classe": c, "linhas_treino_unicas": len(tr_c), "linhas_teste": n_linhas_te, "unicas_teste": n_unicas_te, "usadas_por_dia": m, **{f"recall_validacao_{MODELOS[q]['rotulo']}": REC_VAL[q][c] for q in MODELOS}, **{f"recall_teste_{MODELOS[q]['rotulo']}": REC_TE[q][c] for q in MODELOS}}
        if m < MIN_LINHAS_ADV:
            linha.update({"auc_adversarial": np.nan, "auc_controle": np.nan, "top_atributos": "(poucos vetores únicos: classe pulada)"}); linhas.append(linha); barra.avancar(1, f"{c}: pulada ({m} linhas únicas)"); continue
        A, B = d_tr.iloc[:m].to_numpy(float), d_te[:m]
        auc, X, d = auc_dominio(A, B, SEMENTE)
        mc = len(d_tr) // 2; Ac, Bc = d_tr.iloc[:mc].to_numpy(float), d_tr.iloc[mc:2 * mc].to_numpy(float)       # controle: duas metades do TREINO fingindo ser dois dias
        auc_c = auc_dominio(Ac, Bc, SEMENTE + 1)[0] if mc >= MIN_LINHAS_ADV else np.nan; linha["n_controle_por_metade"] = mc if mc >= MIN_LINHAS_ADV else np.nan
        imp = rf_dom(SEMENTE).fit(X, d).feature_importances_; top = np.argsort(imp)[::-1][:8]
        linha.update({"auc_adversarial": auc, "auc_controle": auc_c, "top_atributos": ", ".join(CM[i] for i in top[:5])})
        ADV_FEAT[c] = pd.DataFrame({"atributo": [CM[i] for i in top], "importancia": imp[top], "KS_treino_x_teste": [ks_2samp(A[:, i], B[:, i]).statistic for i in top],
                                    **{f"treino_{n_}": np.percentile(A[:, top], p_, axis=0) for n_, p_ in (("p10", 10), ("p50", 50), ("p90", 90))}, **{f"teste_{n_}": np.percentile(B[:, top], p_, axis=0) for n_, p_ in (("p10", 10), ("p50", 50), ("p90", 90))}})
        linhas.append(linha); barra.avancar(1, f"{c}: AUC {auc:.3f} (controle {auc_c:.3f}) | {m:,} vetores por dia | {time.perf_counter() - t0:.0f} s"); del A, B, X, Ac, Bc; gc.collect()
    barra.fechar(); ADV = pd.DataFrame(linhas)
    ADV["gravidade"] = [sev(a, REGRAS["auc_adversarial"]) if (np.isfinite(a) and u >= REGRAS["min_linhas_unicas_adversarial"]) else "n/d" for a, u in zip(ADV.auc_adversarial, ADV.unicas_teste)]
    print("\n== Validação adversarial por classe (ROC-AUC de distinguir dia do treino × dia do teste; controle = treino × treino, esperado ~0,5) ==")
    display(ADV.drop(columns=["top_atributos"]).round(4))
    for c, tab_f in ADV_FEAT.items():
        if ADV.set_index("classe").loc[c, "gravidade"] in ("moderada", "grande"):
            print(f"\n-- {c}: os atributos que mais separam os dois dias (importância no classificador de dia, KS e percentis 10/50/90 em cada dia) --"); display(tab_f.round(4))
    try:
        import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
        d_ok = ADV[ADV.auc_adversarial.notna()].reset_index(drop=True)
        if len(d_ok):
            FIG_ADV, ax = plt.subplots(figsize=(8, 0.6 * len(d_ok) + 1.8)); y_ = np.arange(len(d_ok))
            ax.barh(y_, d_ok.auc_adversarial, color="C0", label="treino × teste"); ax.plot(d_ok.auc_controle, y_, "kD", label="controle (treino × treino)")
            for v_ in REGRAS["auc_adversarial"]: ax.axvline(v_, color="gray", ls="--", lw=0.8)
            ax.axvline(0.5, color="gray", lw=0.8); ax.set_yticks(y_); ax.set_yticklabels(d_ok.classe); ax.set_xlim(0.4, 1.02); ax.invert_yaxis(); ax.set_xlabel("ROC-AUC do classificador de dia (0,5 = indistinguível)")
            ax.set_title("Validação adversarial por classe"); ax.legend(loc="lower right"); FIG_ADV.tight_layout(); plt.close(FIG_ADV)
    except ImportError: print("(matplotlib não está instalado: gráfico não gerado)")
else: print("Validação adversarial desligada (FAZER_ADV = False ou vetores do teste ausentes).")
T_SEC["5.1_adversarial"] = time.perf_counter() - t_sec0


Validação adversarial: concluído em 1min 12s

== Validação adversarial por classe (ROC-AUC de distinguir dia do treino × dia do teste; controle = treino × treino, esperado ~0,5) ==


,classe,linhas_treino_unicas,linhas_teste,unicas_teste,usadas_por_dia,recall_validacao_RF ajustado,recall_validacao_RF padrão,recall_validacao_SVM ajustado,recall_validacao_SVM padrão,recall_teste_RF ajustado,recall_teste_RF padrão,recall_teste_SVM ajustado,recall_teste_SVM padrão,n_controle_por_metade,auc_adversarial,auc_controle,gravidade
0,BENIGN,49974,56216,45122,20000,0.9993,0.9990,0.9983,0.9959,0.9806,0.9824,0.9850,0.9884,20000,0.9467,0.4966,grande
1,LDAP,30099,112491,7377,7377,0.9784,0.9646,0.9560,0.9507,0.9926,0.9632,0.9823,0.9847,15049,0.8455,0.5017,grande
2,MSSQL,49979,854242,164156,20000,0.8884,0.8804,0.9179,0.8984,0.3411,0.3681,0.3748,0.3835,20000,0.7352,0.5015,moderada
3,NetBIOS,19388,91175,7507,7507,0.8458,0.7630,0.5441,0.5380,0.9987,0.9990,0.9987,0.9964,9694,0.9633,0.4989,grande
4,Syn,49972,4887060,524064,20000,0.9790,0.9550,0.9912,0.9729,0.1617,0.2445,0.4736,0.5273,20000,1.0000,0.4988,grande
5,UDP,50000,1214746,1118112,20000,0.9409,0.9180,0.9884,0.9816,0.9081,0.8744,0.9736,0.9596,20000,0.7317,0.5021,moderada
6,UDPLag,49946,557,300,300,0.8254,0.8379,0.7014,0.7033,0.3573,0.4039,0.3465,0.0000,20000,0.9937,0.4996,grande



-- BENIGN: os atributos que mais separam os dois dias (importância no classificador de dia, KS e percentis 10/50/90 em cada dia) --


,atributo,importancia,KS_treino_x_teste,treino_p10,treino_p50,treino_p90,teste_p10,teste_p50,teste_p90
0,min_seg_size_forward,0.1308,0.1998,20.0,20.0,3.200000e+01,20.0,20.0,3.200000e+01
1,init_win_bytes_forward,0.0629,0.1082,-1.0,252.0,1.652500e+04,-1.0,256.0,2.920000e+04
2,fwd_packet_length_max,0.0544,0.1752,0.0,35.0,5.170000e+02,0.0,35.0,5.170000e+02
3,fwd_header_length,0.0520,0.0947,20.0,40.0,3.440000e+02,20.0,64.0,6.560000e+02
4,total_length_of_fwd_packets,0.0505,0.1693,0.0,70.0,1.650000e+03,0.0,72.0,2.524000e+03
5,flow_iat_max,0.0450,0.1197,164.0,20996.0,9.980463e+06,80.0,25473.5,1.020695e+07
6,avg_fwd_segment_size,0.0447,0.1752,0.0,31.0,8.873580e+01,0.0,31.0,1.050625e+02
7,max_packet_length,0.0440,0.1766,0.0,58.0,1.460000e+03,0.0,51.0,1.641000e+03



-- LDAP: os atributos que mais separam os dois dias (importância no classificador de dia, KS e percentis 10/50/90 em cada dia) --


,atributo,importancia,KS_treino_x_teste,treino_p10,treino_p50,treino_p90,teste_p10,teste_p50,teste_p90
0,fwd_header_length,0.1217,0.0744,0.000000e+00,1.443000e+03,2.896000e+03,-4.0000,2.700000e+02,2.880000e+03
1,min_seg_size_forward,0.1088,0.0816,0.000000e+00,3.480000e+02,1.447000e+03,-1.0000,7.400000e+01,1.428000e+03
2,flow_bytes_s,0.0782,0.2046,1.182329e+07,7.009524e+07,2.848000e+09,2677.6636,5.975510e+07,2.832000e+09
3,flow_iat_min,0.0772,0.3036,1.000000e+00,5.000000e+00,7.900000e+01,0.0000,1.000000e+00,5.000000e+01
4,flow_duration,0.0760,0.1675,1.000000e+00,3.900000e+01,2.144000e+02,1.0000,7.000000e+00,9.360000e+02
5,flow_iat_std,0.0726,0.2119,0.000000e+00,0.000000e+00,0.000000e+00,0.0000,0.000000e+00,3.376943e+02
6,total_length_of_fwd_packets,0.0680,0.1609,2.528000e+03,2.928000e+03,2.944000e+03,2000.0000,2.896000e+03,5.888000e+03
7,flow_iat_max,0.0676,0.1666,1.000000e+00,3.900000e+01,1.948000e+02,1.0000,7.000000e+00,7.190000e+02


... (6 outputs left out)


-- UDP: os atributos que mais separam os dois dias (importância no classificador de dia, KS e percentis 10/50/90 em cada dia) --


,atributo,importancia,KS_treino_x_teste,treino_p10,treino_p50,treino_p90,teste_p10,teste_p50,teste_p90
0,fwd_header_length,0.2773,0.0984,0.0000,80.0000,700.0000,-2.0000,72.0000,144.0000
1,min_seg_size_forward,0.2550,0.1438,-1.0000,8.0000,20.0000,-1.0000,8.0000,20.0000
2,flow_iat_min,0.0587,0.0698,0.0000,1.0000,1.0000,0.0000,1.0000,1.0000
3,flow_iat_max,0.0519,0.0876,105044.0000,109503.5000,110796.0000,104988.4000,109780.0000,110904.0000
4,flow_duration,0.0487,0.0931,105083.8000,212118.0000,219512.5000,105081.0000,213725.0000,220674.4000
5,flow_iat_mean,0.0467,0.0903,34996.9667,42425.5000,43912.3000,34932.3000,42745.6000,44138.4800
6,flow_bytes_s,0.0459,0.0868,9500.9239,9850.9617,13586.0451,9449.7939,9775.6241,13607.3862
7,flow_iat_std,0.0454,0.0876,56378.5747,59824.5210,62864.2072,56348.5310,59514.0372,62825.2360



-- UDPLag: os atributos que mais separam os dois dias (importância no classificador de dia, KS e percentis 10/50/90 em cada dia) --


,atributo,importancia,KS_treino_x_teste,treino_p10,treino_p50,treino_p90,teste_p10,teste_p50,teste_p90
0,flow_packets_s,0.0908,0.6933,0.1554,2.448000e-01,3.731680e+01,6.4461,3034.5992,6.666667e+05
1,flow_iat_std,0.0870,0.6933,59280.1971,7.371666e+06,1.218036e+07,0.0000,603.5305,3.906563e+05
2,flow_iat_mean,0.0862,0.6933,35730.1333,4.702424e+06,7.919056e+06,2.0000,442.5000,2.160313e+05
3,min_seg_size_forward,0.0846,0.6267,0.0000,2.000000e+01,2.000000e+01,20.0000,32.0000,3.200000e+01
4,flow_iat_min,0.0819,0.6800,0.0000,1.000000e+00,1.000000e+00,1.0000,2.0000,4.000000e+00
5,flow_iat_max,0.0815,0.6667,107187.3000,1.535167e+07,2.631358e+07,3.0000,1182.0000,2.090767e+06
6,active_mean,0.0526,0.6500,0.0000,1.000000e+00,5.100000e+01,0.0000,0.0000,0.000000e+00
7,init_win_bytes_forward,0.0407,0.6067,-1.0000,5.840000e+03,5.840000e+03,-1.0000,233.0000,2.920000e+04


## 6. Veredito (pelas regras gravadas no início)
Aplica as regras e escreve um **texto sugerido para o TCC** (revise antes de usar).

In [12]:
# ---------- 6.1 Veredito ----------
t_sec0 = time.perf_counter()
pt = lambda x, nd=2: f"{x:.{nd}f}".replace(".", ",")                       # número com vírgula decimal (para o texto do TCC)
perm_sit = {r.modelo: ("pequena" if r.situacao.startswith("ok") else "grande") for r in PERM_RES.itertuples()} if len(PERM_RES) else {}
adv_validas = ADV[ADV.gravidade.isin(["pequena", "moderada", "grande"])] if len(ADV) else pd.DataFrame()
if len(adv_validas): adv_pior, adv_cls = pior(*adv_validas.gravidade), adv_validas.sort_values("auc_adversarial", ascending=False).iloc[0]
else: adv_pior, adv_cls = "n/d", None
VEREDITO = {}
for chave, M in MODELOS.items():
    rot = M["rotulo"]; L = LAC.loc[rot]; T = TAB.loc[rot]; eh_rf = M["familia"] == "RF"
    q1 = pior(L["Q1 acc.bal."], L["Q1 ROC-AUC"], perm_sit.get(rot, "n/d")); q2_modelo = pior(L["Q2 ROC-AUC"], L["Q2 F1 macro"], L["Q2 acc.bal."], L["Q2 recall classe"]); q2 = "n/d" if q2_modelo == "n/d" else pior(q2_modelo, adv_pior)
    lacuna = f"lacuna treino−validação: acurácia balanceada {T['lacuna_acc_bal']:+.4f}, ROC-AUC {T['lacuna_roc_auc']:+.5f}"; cv_val = f"CV−validação (acurácia balanceada) {T['cv_acc_bal'] - T['validacao_acc_bal']:+.4f}"
    queda = f"validação→teste: ROC-AUC {-T['queda_roc_auc_teste']:+.4f}, F1 macro {-T['queda_f1_macro_teste']:+.4f}, acurácia balanceada {-T['queda_acc_bal_teste']:+.4f}, maior queda de recall {-T['queda_recall_classe']:+.3f} ({T['classe_maior_queda_recall']})"
    adv_txt = f" (classe mais distinta: {adv_cls['classe']}, ROC-AUC adversarial {adv_cls['auc_adversarial']:.3f}; atributos: {adv_cls['top_atributos']})" if adv_cls is not None else ""
    if q1 in ("moderada", "grande"):
        notas = []
        if eh_rf and q1 != "n/d" and L["Q1 acc.bal."] in ("moderada", "grande"): notas.append("esperado para árvores sem limite de profundidade, que acertam o treino por construção; o que importa é a validação cruzada concordar com a validação e o embaralhamento ficar em 0,5")
        if perm_sit.get(rot) == "grande": notas.append("o embaralhamento de rótulos fugiu de 0,5 (investigar vazamento)")
        leitura = "HÁ SINAL de lacuna treino−validação (" + lacuna + "; " + cv_val + ")" + ((" — " + "; ".join(notas)) if notas else "")
    elif q2_modelo == "n/d": leitura = "sem sinal de decorar o treino; a comparação com o teste não pôde ser feita (arquivos do teste ausentes)"
    elif q2 == "pequena": leitura = "SEM SINAL de sobreajuste de nenhum tipo: " + lacuna + "; " + queda
    elif adv_pior in ("moderada", "grande") and q2_modelo in ("moderada", "grande"): leitura = f"sem sinal de decorar o treino ({lacuna}; {cv_val}); a queda no teste ({queda}) é COMPATÍVEL com mudança de distribuição entre os dias" + adv_txt
    elif q2_modelo in ("moderada", "grande"): leitura = f"sem sinal de decorar o treino ({lacuna}); há queda no teste ({queda}), mas a validação adversarial NÃO mostra mudança relevante nos atributos: a causa precisa de outra investigação"
    else: leitura = f"sem sinal de decorar o treino ({lacuna}); queda no teste pequena; a validação adversarial indica mudança entre os dias" + adv_txt + ", sem efeito relevante neste modelo"
    VEREDITO[rot] = {"Q1_decorar_o_treino": q1, "Q2_generalizar_outro_dia": q2, "Q2_so_modelo": q2_modelo, "leitura": leitura}
    ganho = GANHO_50_100.get(rot); VEREDITO[rot]["curva_ganho_50_100"] = None if ganho is None else {"ganho_roc_auc": ganho, "situacao": sev(ganho, REGRAS["curva_ganho_50_100"])}
RES = pd.DataFrame({r: {"Q1 decorar o treino": v["Q1_decorar_o_treino"], "Q2 outro dia": v["Q2_generalizar_outro_dia"]} for r, v in VEREDITO.items()}).T
print("== Gravidade dos sinais, por modelo ==\n"); display(RES)
for r, v in VEREDITO.items(): print(f"• {r}: {v['leitura']}")
print("\n--- Texto sugerido para o TCC (revise antes de usar) ---")
q1_ok = [r for r, x in VEREDITO.items() if x["Q1_decorar_o_treino"] == "pequena"]; q1_alerta = [r for r, x in VEREDITO.items() if x["Q1_decorar_o_treino"] in ("moderada", "grande")]
gap_cv_val = float((TAB["cv_acc_bal"] - TAB["validacao_acc_bal"]).abs().max()); perm_ok = bool(perm_sit) and all(x == "pequena" for x in perm_sit.values())
TEXTO = (f"Para investigar sobreajuste, comparamos o desempenho dos {len(MODELOS)} modelos multiclasse nas linhas de treino, na validação cruzada, em uma validação separada do mesmo dia e no teste de outro dia, "
         f"e complementamos com curvas de aprendizagem, um controle com rótulos embaralhados e uma validação adversarial por classe (dia do treino × dia do teste). "
         + ("A lacuna entre o desempenho no treino e na validação foi pequena em todos os modelos. " if not q1_alerta else
            (f"A lacuna entre o desempenho no treino e na validação foi pequena em {', '.join(q1_ok)} e relevante em {', '.join(q1_alerta)}" if q1_ok else f"A lacuna entre o desempenho no treino e na validação foi relevante em {', '.join(q1_alerta)}")
            + (" (nos Random Forests, isso é esperado para árvores sem limite de profundidade, que acertam quase todas as linhas que treinaram). " if any("RF" in r for r in q1_alerta) else ". "))
         + (f"A validação cruzada e a validação do mesmo dia concordaram entre si (diferença máxima de {pt(gap_cv_val, 3)} na acurácia balanceada)" if gap_cv_val < 0.02 else f"A validação cruzada e a validação do mesmo dia diferiram em até {pt(gap_cv_val, 3)} na acurácia balanceada")
         + (", e o controle com rótulos embaralhados ficou em torno do acaso. " if perm_ok else (". " if not perm_sit else ", mas o controle com rótulos embaralhados fugiu do acaso em ao menos um modelo (investigar). "))
         + (f"{pt(100 * OV_FRAC, 2)}% das linhas da validação têm vetor de atributos idêntico ao de uma linha do ajuste ({pt(OV_INFO['linhas'], 0)} linhas; {pt(OV_INFO['conflitos'], 0)} com rótulo diferente)" + (", o que torna a validação do mesmo dia otimista. " if OV_SEV in ("moderada", "grande") else ", ou seja, a validação é praticamente independente do ajuste quanto a vetores repetidos. "))
         + (f"A queda de desempenho no teste (de outro dia) é compatível com mudança de distribuição entre os dias: a validação adversarial distinguiu treino e teste com ROC-AUC de até {pt(adv_cls['auc_adversarial'])} (classe {adv_cls['classe']}), "
            f"com {adv_cls['top_atributos']} entre os atributos que mais mudam. " if adv_cls is not None and adv_pior in ("moderada", "grande") else "")
         + "Esses resultados são evidências descritivas, obtidas em uma única partição de teste com muitas linhas repetidas; não provam a ausência de sobreajuste, e a validação do mesmo dia do treino não é capaz de detectar o ajuste às condições de um dia específico.")
print(TEXTO)
T_SEC["6.1_veredito"] = time.perf_counter() - t_sec0


== Gravidade dos sinais, por modelo ==



,Q1 decorar o treino,Q2 outro dia
RF ajustado,moderada,grande
RF padrão,grande,grande
SVM ajustado,grande,grande
SVM padrão,pequena,grande


• RF ajustado: HÁ SINAL de lacuna treino−validação (lacuna treino−validação: acurácia balanceada +0.0199, ROC-AUC +0.00288; CV−validação (acurácia balanceada) +0.0005) — esperado para árvores sem limite de profundidade, que acertam o treino por construção; o que importa é a validação cruzada concordar com a validação e o embaralhamento ficar em 0,5
• RF padrão: HÁ SINAL de lacuna treino−validação (lacuna treino−validação: acurácia balanceada +0.0885, ROC-AUC +0.00971; CV−validação (acurácia balanceada) +0.0015) — esperado para árvores sem limite de profundidade, que acertam o treino por construção; o que importa é a validação cruzada concordar com a validação e o embaralhamento ficar em 0,5
• SVM ajustado: HÁ SINAL de lacuna treino−validação (lacuna treino−validação: acurácia balanceada -0.0014, ROC-AUC -0.00003; CV−validação (acurácia balanceada) -0.0020) — o embaralhamento de rótulos fugiu de 0,5 (investigar vazamento)
• SVM padrão: sem sinal de decorar o treino (lacuna treino−valida

## 7. Gravação
Grava as tabelas, os gráficos, o veredito e o texto numa pasta nova `diagnostico_sobreajuste_multiclasse/diag_<data>`. Cada arquivo só é criado se ainda não existir.

In [13]:
# ---------- 7.1 Gravação ----------
t_sec0 = time.perf_counter()
gravar_csv(TAB.reset_index(), PASTA_DIAG / "tabela_treino_cv_validacao_teste.csv", index=False); gravar_csv(LAC.reset_index(), PASTA_DIAG / "lacunas_e_quedas.csv", index=False); gravar_csv(rec_tab.reset_index(names="classe"), PASTA_DIAG / "recall_por_classe_validacao_teste.csv", index=False)
if len(CURVA): gravar_csv(CURVA, PASTA_DIAG / "curva_aprendizagem_sorteios.csv", index=False); gravar_csv(CURVA_RES, PASTA_DIAG / "curva_aprendizagem.csv", index=False)
if len(PERM): gravar_csv(PERM, PASTA_DIAG / "embaralhamento_repeticoes.csv", index=False); gravar_csv(PERM_RES, PASTA_DIAG / "embaralhamento_resumo.csv", index=False)
gravar_csv(OVERLAP_TAB.reset_index(), PASTA_DIAG / "sobreposicao_ajuste_validacao.csv", index=False)
if len(ADV):
    gravar_csv(ADV, PASTA_DIAG / "adversarial_por_classe.csv", index=False)
    for c, tab_f in ADV_FEAT.items(): gravar_csv(tab_f, PASTA_DIAG / f"adversarial_atributos_{c}.csv", index=False)
for fig, nome in ((FIG_CURVA, "curva_aprendizagem.png"), (FIG_ADV, "adversarial_por_classe.png")):
    if fig is not None: gravar(PASTA_DIAG / nome, lambda p_, f=fig: f.savefig(p_, dpi=130))
gravar_json({"veredito": VEREDITO, "sobreposicao_ajuste_validacao": OV_INFO, "regras": REGRAS, "modelos": {k: {"rotulo": M["rotulo"], "descricao": M["descricao"]} for k, M in MODELOS.items()}, "ganho_curva_50_100": GANHO_50_100, "adversarial_pior_gravidade": adv_pior,
             "parametros": {"n_treino_svm_aval": N_TREINO_SVM_AVAL, "n_adv": N_ADV, "reps_curva_rf": REPS_CURVA_RF, "n_perm_rf": N_PERM_RF, "n_perm_svm": N_PERM_SVM, "semente": SEMENTE}, "data": datetime.now().isoformat(timespec="seconds")}, PASTA_DIAG / "veredito.json")
gravar(PASTA_DIAG / "texto_tcc.txt", lambda p_: p_.write_text(TEXTO, encoding="utf-8"))
T_SEC["7.1_gravacao"] = time.perf_counter() - t_sec0
gravar_csv(pd.DataFrame({"etapa": list(T_SEC), "segundos": [round(v, 1) for v in T_SEC.values()]}), PASTA_DIAG / "tempos_etapas.csv", index=False)
print(f"✔ Resultados gravados em {PASTA_DIAG}")
for f in sorted(PASTA_DIAG.iterdir()): print(f"   {f.name:46s} {f.stat().st_size / 2**10:>9,.1f} KiB")


✔ Resultados gravados em C:\Users\Bruno\downloads\tcc_codigo\railway-ch\artefatos_modelos\diagnostico_sobreajuste_multiclasse\diag_20261111_233152
   adversarial_atributos_BENIGN.csv                     0.8 KiB
   adversarial_atributos_LDAP.csv                       0.9 KiB
   adversarial_atributos_MSSQL.csv                      0.9 KiB
   adversarial_atributos_NetBIOS.csv                    1.0 KiB
   adversarial_atributos_Syn.csv                        0.8 KiB
   adversarial_atributos_UDP.csv                        1.0 KiB
   adversarial_atributos_UDPLag.csv                     1.0 KiB
   adversarial_por_classe.csv                           2.3 KiB
   adversarial_por_classe.png                          40.2 KiB
   curva_aprendizagem.csv                               3.4 KiB
   curva_aprendizagem.png                             252.5 KiB
   curva_aprendizagem_sorteios.csv                      5.1 KiB
   embaralhamento_repeticoes.csv                        1.0 KiB
   embaralhamento_res

## 8. Conferências
Confere que a validação recalculada reproduz o ROC-AUC gravado pelos notebooks dos modelos, que as regras foram gravadas antes dos resultados, que o controle adversarial ficou em torno de 0,5 e que **nenhum arquivo das pastas dos modelos foi alterado**.

In [15]:
# ---------- 8.1 Conferências ----------
ok = []
def verifica(nome, cond, detalhe=""):
    ok.append(bool(cond)); print(("✔" if cond else "✘"), nome, detalhe)
verifica("Os quatro modelos foram diagnosticados e as tabelas têm todas as origens (treino, CV, validação, teste)", len(MODELOS) == 4 and len(TAB) == 4 and not TAB[[c for c in TAB.columns if c.split("_")[0] in ("treino", "cv", "validacao", "teste") and c.endswith("roc_auc")]].isna().any().any())
verifica("As regras de decisão foram gravadas ANTES de qualquer resultado", T_REGRAS < min(f.stat().st_mtime_ns for f in PASTA_DIAG.iterdir() if f.name != "regras_decisao.json") and (PASTA_DIAG / "regras_decisao.json").exists())
# o desempenho na validação recalculado aqui precisa reproduzir o gravado pelo notebook de cada modelo (mesmo modelo, mesmos dados)
def val_gravado(chave):
    M = MODELOS[chave]
    if chave == "svm_padrao": g = pd.read_csv(PASTA_SVM / "resultados_grade.csv", keep_default_na=False); return float(g[g["config"].astype(str) == M["cfg_grade"]].iloc[0]["val_roc_auc"])
    pasta = {"rf_ajustado": PASTA_RF, "rf_padrao": PASTA_RF_PAD, "svm_ajustado": PASTA_SVM}[chave]; return float(json.loads((pasta / "metricas_validacao.json").read_text(encoding="utf-8"))["sem_pesos"]["roc_auc_ovr_macro"])
dif = {k: abs(TAB.loc[MODELOS[k]["rotulo"], "validacao_roc_auc"] - val_gravado(k)) for k in MODELOS}
verifica("A validação recalculada reproduz o ROC-AUC gravado pelos notebooks dos modelos (05m, 07m, 11m)", all(v < 1e-6 for v in dif.values()), "(diferença máxima " + f"{max(dif.values()):.1e})")
verifica("O teste mostrado é o gravado (as linhas por classe somam o total do teste, nos 4 modelos)", len({sum(N_TE[k].values()) for k in MODELOS}) == 1 and all(sum(N_TE[k].values()) == json.loads(MODELOS[k]["json_teste"].read_text(encoding="utf-8"))["n_teste"] for k in MODELOS))
if len(CURVA_RES):
    dif_curva = {MODELOS[k]["rotulo"]: abs(float(CURVA_RES[(CURVA_RES.modelo == MODELOS[k]["rotulo"]) & (CURVA_RES.fracao == 1.0)].validacao_roc_auc.iloc[0]) - TAB.loc[MODELOS[k]["rotulo"], "validacao_roc_auc"]) for k in MODELOS if MODELOS[k]["familia"] == "RF" and (CURVA_RES.modelo == MODELOS[k]["rotulo"]).any()}
    verifica("Curva de aprendizagem: o ponto de 100% do RF reproduz a validação da tabela (tolerância 1e-6: o RF em vários núcleos pode desempatar escores iguais em outra ordem, o que muda o ROC-AUC na 9ª casa decimal)", all(v < 1e-6 for v in dif_curva.values()), "(diferença máxima " + f"{max(dif_curva.values()):.1e})")
verifica("Sobreposição ajuste × validação: as contagens fecham (idênticas ≤ linhas; conflitos ≤ idênticas; soma dos grupos = total)", int(OVERLAP_TAB.loc["todas", "com vetor idêntico ao de uma linha do ajuste"]) <= len(y_va) and OV_INFO["conflitos"] <= OV_INFO["linhas"] and int(OVERLAP_TAB.drop(index="todas")["com vetor idêntico ao de uma linha do ajuste"].sum()) == OV_INFO["linhas"], f"({OV_INFO['linhas']:,} de {len(y_va):,})")
if len(PERM): verifica("Embaralhamento: houve repetições para todos os modelos pedidos", len(PERM_RES) >= 1 and bool((PERM_RES.repeticoes >= 1).all()))
if len(ADV):
    valid = ADV[ADV.auc_controle.notna()]; tol = np.maximum(0.03, 4 * np.sqrt((2 * valid.n_controle_por_metade + 1) / (12 * valid.n_controle_por_metade ** 2))) if len(valid) else None        # tolerância: 4 desvios do acaso (mínimo 0,03)
    verifica("Validação adversarial: o controle (treino × treino) fica em torno de 0,5 em todas as classes (dentro de 4 desvios do acaso)", bool(((valid.auc_controle - 0.5).abs() <= tol).all()) if len(valid) else True, f"(controle entre {valid.auc_controle.min():.3f} e {valid.auc_controle.max():.3f}; tolerância mínima {tol.min():.3f})" if len(valid) else "")
    verifica("Validação adversarial: as linhas do teste por classe fecham com as do 12m", int(ADV.linhas_teste.sum()) == sum(N_TE["svm_ajustado"].values()), f"({int(ADV.linhas_teste.sum()):,})")
_depois = {**foto(PASTA_RF, "rf_aj"), **foto(PASTA_RF_PAD, "rf_pad"), **foto(PASTA_SVM, "svm")}; _mexidos = [n for n, v in SNAP_ANTES.items() if _depois.get(n) != v]
verifica(f"Nenhum dos {len(SNAP_ANTES)} arquivos que já existiam nas pastas dos modelos foi alterado, sobrescrito ou apagado", not _mexidos, f"({_mexidos[:3]})" if _mexidos else "")
verifica("Arquivos esperados gravados", all((PASTA_DIAG / f).exists() for f in ("regras_decisao.json", "tabela_treino_cv_validacao_teste.csv", "lacunas_e_quedas.csv", "veredito.json", "texto_tcc.txt", "tempos_etapas.csv")))
assert all(ok), "há checagens reprovadas acima"
print(f"\n✔ {sum(ok)}/{len(ok)} checagens aprovadas. Resultados em {PASTA_DIAG}"); print(f"\n⏱ Tempo total do notebook (etapas medidas): {fmt_tempo(float(sum(T_SEC.values())))}")
print("\nPróximo passo: me mande o texto desta saída (e, se quiser, os arquivos adversarial_por_classe.csv, adversarial_atributos_*.csv, tabela_treino_cv_validacao_teste.csv e os gráficos).")

✔ Os quatro modelos foram diagnosticados e as tabelas têm todas as origens (treino, CV, validação, teste) 
✔ As regras de decisão foram gravadas ANTES de qualquer resultado 
✔ A validação recalculada reproduz o ROC-AUC gravado pelos notebooks dos modelos (05m, 07m, 11m) (diferença máxima 3.2e-09)
✔ O teste mostrado é o gravado (as linhas por classe somam o total do teste, nos 4 modelos) 
✔ Curva de aprendizagem: o ponto de 100% do RF reproduz a validação da tabela (tolerância 1e-6: o RF em vários núcleos pode desempatar escores iguais em outra ordem, o que muda o ROC-AUC na 9ª casa decimal) (diferença máxima 3.7e-09)
✔ Sobreposição ajuste × validação: as contagens fecham (idênticas ≤ linhas; conflitos ≤ idênticas; soma dos grupos = total) (1,207 de 59,918)
✔ Embaralhamento: houve repetições para todos os modelos pedidos 
✔ Validação adversarial: o controle (treino × treino) fica em torno de 0,5 em todas as classes (dentro de 4 desvios do acaso) (controle entre 0.497 e 0.502; tolerância